# Markt­konfigurations-Auswertung mit erhaltener Original-Pipeline

Dieses Notebook übernimmt die **Einlese- und Merge-Logik des ursprünglichen Notebooks**. Unverändert enthalten sind insbesondere:

- Discovery der Reporting-Roots und Experiment-Instanzen,
- Prüfung der zentralen Config-Signaturen,
- Zusammenführung gleichnamiger Experimentordner über mehrere Reporting-Roots,
- Behandlung doppelter Run-IDs,
- Ermittlung der `run_records`,
- Auswahl von `Data/agent_sc_level_1.csv`,
- Berechnung der Agenten- und Systemmetriken.

Erst **nachdem `run_metrics` durch diese Original-Pipeline erzeugt wurde**, beginnt die neue Auswertung je Marktkonfiguration. Es gibt keine Plot-, Bild- oder Chart-Generierung.

In [ ]:
from pathlib import Path
import json
import re
import warnings
import zipfile
import math
import hashlib
from itertools import combinations
from typing import Any, Iterable

import numpy as np
import pandas as pd

from IPython.display import display, HTML
from scipy import stats
from scipy.stats import t as student_t
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)


## 1. Konfigurierbare Pfade und Einstellungen

Der Standardmodus bleibt die Auswertung des hochgeladenen ZIP-Archivs. Über `USE_ZIP_INPUT = False` kann dieselbe ursprüngliche Discovery- und Merge-Pipeline alternativ wieder direkt auf frei konfigurierbare `REPORTING_ROOTS` im Projekt angewendet werden.

In [ ]:
# =============================================================================
# ZENTRALE KONFIGURATION
# =============================================================================
# In dieser Zelle sollten alle Pfade und Analyseparameter angepasst werden.
# Die nachfolgenden Zellen enthalten weiterhin die ursprüngliche Discovery-,
# Config-Prüfungs-, Merge- und Run-Einleselogik.
# =============================================================================


# -----------------------------------------------------------------------------
# Projektpfad
# -----------------------------------------------------------------------------
# Wird insbesondere für den manuellen Reporting-Root-Modus verwendet.
# Passe diesen Pfad an, wenn dein Projekt woanders liegt.
PROJECT_ROOT = Path(
    "/home/jupyter-dom30542/designingeffectivecollaborativelearningsystems"
)

PROJECT_ROOT = Path("./")

print("Current working directory:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

reporting_dir = PROJECT_ROOT / "Reporting" / "00_MultiProduct"
print("Reporting directory:", reporting_dir.resolve())
print("Reporting directory exists:", reporting_dir.exists())
# -----------------------------------------------------------------------------
# Eingabemodus
# -----------------------------------------------------------------------------
# True:
#   Das ZIP-Archiv wird entpackt. Danach werden innerhalb von EXTRACT_BASE
#   automatisch alle Verzeichnisse erkannt, die direkt Konfigurationsordner
#   mit config.json enthalten.
#
# False:
#   Die weiter unten konfigurierten manuellen REPORTING_ROOTS werden verwendet.
#   Damit verhält sich die Pfadkonfiguration wie im ursprünglichen Notebook.
USE_ZIP_INPUT = False


# -----------------------------------------------------------------------------
# ZIP-Input und Extraktion
# -----------------------------------------------------------------------------
# Diese Pfade bleiben standardmäßig auf der hochgeladenen ZIP-Datei.
INPUT_ZIP = Path(
    "/mnt/data/real_world_multi_product_with_fusion_all_runs(1).zip"
)
EXTRACT_BASE = Path(
    "/mnt/data/real_world_extracted_preserved_logic"
)

# Das Archiv wird nur entpackt, wenn EXTRACT_BASE leer ist.
# Zum erneuten Entpacken kann ein neuer EXTRACT_BASE-Pfad vergeben oder der
# vorhandene Ordner außerhalb dieses Notebooks gelöscht werden.
EXTRACT_ZIP_IF_TARGET_EMPTY = False


# -----------------------------------------------------------------------------
# Manuell konfigurierbare Reporting-Roots
# -----------------------------------------------------------------------------
# Dieser Block wird nur verwendet, wenn USE_ZIP_INPUT = False ist.
#
# Jeder Eintrag muss direkt die Konfigurationsordner enthalten, also z. B.
# chronos_zero_shot_001/, timesfm_zero_shot_001/, usw.
#
# Die Schreibweise `synthethic` bleibt aus Gründen der Rückwärtskompatibilität
# zum ursprünglichen Notebook erhalten.
synthethic = True

SYNTHETIC_REPORTING_ROOTS = [
    PROJECT_ROOT / "Reporting" / "zero_shot_multi_product_runs_0_to_10_lambda_075_tau_0_2026_07_17_13_26_27",

    PROJECT_ROOT / "Reporting" / "without_zero_shot_multi_product_runs_0_lambda_075_tau_0_2026_07_17_13_51_54",
    PROJECT_ROOT / "Reporting" / "without_zero_shot_multi_product_runs_1_lambda_075_tau_0_2026_07_17_14_35_07",
    PROJECT_ROOT / "Reporting" / "without_zero_shot_multi_product_runs_2_lambda_075_tau_0_2026_07_17_17_05_11",
    PROJECT_ROOT / "Reporting" / "without_zero_shot_multi_product_runs_3_lambda_075_tau_0_2026_07_17_17_07_34",
    PROJECT_ROOT / "Reporting" / "without_zero_shot_multi_product_runs_4_lambda_075_tau_0_2026_07_17_15_14_23",

    PROJECT_ROOT / "Reporting" / "without_zero_shot_multi_product_runs_5_part_1_lambda_075_tau_0_2026_07_18_07_32_03",
    PROJECT_ROOT / "Reporting" / "without_zero_shot_multi_product_runs_5_part_2_lambda_075_tau_0_2026_07_18_07_35_29",
    PROJECT_ROOT / "Reporting" / "without_zero_shot_multi_product_runs_5_part_3_lambda_075_tau_0_2026_07_18_07_39_22",

    PROJECT_ROOT / "Reporting" / "without_zero_shot_multi_product_runs_6_lambda_075_tau_0_2026_07_17_17_24_35",
    PROJECT_ROOT / "Reporting" / "without_zero_shot_multi_product_runs_7_lambda_075_tau_0_2026_07_17_22_10_36",
    PROJECT_ROOT / "Reporting" / "without_zero_shot_multi_product_runs_8_lambda_075_tau_0_2026_07_17_20_24_52",
    PROJECT_ROOT / "Reporting" / "without_zero_shot_multi_product_runs_9_lambda_075_tau_0_2026_07_17_21_03_30",
]


REAL_WORLD_REPORTING_ROOTS = [
    # Frühere, über mehrere Teil-Reports verteilte Real-World-Runs:
    PROJECT_ROOT / "Reporting" / "00_MultiProduct" / "real_world_multiple_products_runs_0_2026_07_12_07_43_50",
    PROJECT_ROOT / "Reporting" / "00_MultiProduct" / "real_world_multiple_products_runs_1_to_4_2026_07_12_07_55_38",
    PROJECT_ROOT / "Reporting" / "00_MultiProduct" / "real_world_multiple_products_runs_5_to_9_2026_07_12_09_07_26",

    # Aktueller zusammengefasster Real-World-Report:
    # PROJECT_ROOT
    # / "Reporting"
    # / "real_world_multi_product_runs_0_to_9_2026_07_15_08_48_39",
]

import json
from collections import defaultdict
from pathlib import Path


def find_parameter(obj, parameter_name, location="root"):
    """Recursively find every key matching parameter_name."""
    matches = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            current_location = f"{location}.{key}"

            if str(key).lower() == parameter_name.lower():
                matches.append((current_location, value))

            matches.extend(
                find_parameter(value, parameter_name, current_location)
            )

    elif isinstance(obj, list):
        for index, value in enumerate(obj):
            matches.extend(
                find_parameter(
                    value,
                    parameter_name,
                    f"{location}[{index}]",
                )
            )

    return matches


values = defaultdict(set)

for reporting_root in SYNTHETIC_REPORTING_ROOTS:
    for config_path in Path(reporting_root).rglob("config.json"):
        with config_path.open(encoding="utf-8") as file:
            config = json.load(file)

        lambda_matches = find_parameter(config, "lam")
        tau_matches = find_parameter(config, "tau")

        print(f"\nCONFIG: {config_path}")
        print("  lambda:", lambda_matches or "NOT FOUND")
        print("  tau:", tau_matches or "NOT FOUND")

        for _, value in lambda_matches:
            values["lambda"].add(str(value))

        for _, value in tau_matches:
            values["tau"].add(str(value))

print("\nUNIQUE LAMBDA VALUES:", sorted(values["lambda"]))
print("UNIQUE TAU VALUES:", sorted(values["tau"]))

# -----------------------------------------------------------------------------
# Automatische Reporting-Root-Erkennung
# -----------------------------------------------------------------------------
# Im ZIP-Modus werden die Roots immer innerhalb von EXTRACT_BASE gesucht.
# Im manuellen Modus kann zusätzlich unter REPORTING_BASE gesucht werden.
AUTO_DISCOVER_REPORTING_ROOTS = False
REPORTING_BASE = PROJECT_ROOT / "Reporting"
REPORTING_ROOT_NAME_PATTERN = None  # z. B. r"^2026_05_"


# -----------------------------------------------------------------------------
# Hilfsfunktion für den ZIP-Modus
# -----------------------------------------------------------------------------
def _contains_direct_config_folders(path: Path) -> bool:
    """True, wenn path direkt Experimentordner mit config.json enthält."""
    path = Path(path)
    if not path.exists() or not path.is_dir():
        return False

    return any(
        child.is_dir() and (child / "config.json").exists()
        for child in path.iterdir()
    )


# -----------------------------------------------------------------------------
# Eingabe vorbereiten und REPORTING_ROOTS festlegen
# -----------------------------------------------------------------------------
if USE_ZIP_INPUT:
    if not INPUT_ZIP.exists():
        raise FileNotFoundError(f"Input ZIP not found: {INPUT_ZIP}")

    EXTRACT_BASE.mkdir(parents=True, exist_ok=True)

    target_is_empty = not any(EXTRACT_BASE.iterdir())
    if EXTRACT_ZIP_IF_TARGET_EMPTY and target_is_empty:
        with zipfile.ZipFile(INPUT_ZIP) as zf:
            zf.extractall(EXTRACT_BASE)

    REPORTING_ROOTS = sorted(
        [
            path
            for path in [EXTRACT_BASE, *EXTRACT_BASE.rglob("*")]
            if _contains_direct_config_folders(path)
        ],
        key=lambda path: str(path),
    )

    if not REPORTING_ROOTS:
        raise RuntimeError(
            "No reporting root was found after ZIP extraction. "
            "A reporting root must directly contain experiment folders "
            "with config.json."
        )

    # Die Roots wurden bereits innerhalb des ZIP-Inhalts erkannt.
    # Eine zusätzliche Suche unter PROJECT_ROOT ist im ZIP-Modus nicht nötig.
    AUTO_DISCOVER_REPORTING_ROOTS = False
    REPORTING_BASE = EXTRACT_BASE

else:
    REPORTING_ROOTS = (
        list(SYNTHETIC_REPORTING_ROOTS)
        if synthethic
        else list(REAL_WORLD_REPORTING_ROOTS)
    )


# -----------------------------------------------------------------------------
# Merge-, Config- und Run-ID-Verhalten
# -----------------------------------------------------------------------------
# Wenn derselbe erste Experimentordner in mehreren Reporting-Roots vorkommt,
# werden die Run-Ordner zu einer Experimentgruppe zusammengeführt.
MERGE_SAME_EXPERIMENT_FOLDER_ACROSS_ROOTS = True

# Bei abweichenden zentralen Config-Feldern abbrechen, statt unterschiedliche
# Experimente zusammenzuführen.
CONFIG_MATCH_STRICT = True

# Verhalten bei doppelten run_ids innerhalb derselben Experimentgruppe:
# "error", "keep_first", "keep_last" oder "keep_all".
DUPLICATE_RUN_ID_POLICY = "keep_first"


# -----------------------------------------------------------------------------
# Ausgabepfade
# -----------------------------------------------------------------------------
# Der übergeordnete Results-Ordner wird automatisch erzeugt, falls er noch
# nicht existiert.
RESULTS_ROOT = PROJECT_ROOT / "Results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# Optionaler Name des Ergebnisordners.
# - String setzen, z. B. "Auswertung_Juli_2026", um einen festen Namen zu nutzen.
# - None oder "" erzeugt automatisch:
#     Results_synth_<true|false>_lambda_<Wert(e)>_tau_<Wert(e)>
#   Lambda und Tau werden dafür aus den analysierten config.json-Dateien gelesen.
RESULTS_FOLDER_NAME = None

OUTPUT_XLSX_FILENAME = (
    "market_configuration_evaluation_summaries_preserved_merge.xlsx"
)
ANALYSIS_CONFIG_FILENAME = "analysis_config.json"

# Vorläufige Werte für Rückwärtskompatibilität. Sobald die Configs und das
# tatsächliche Auswertungsfenster bekannt sind, wird der endgültige
# Ergebnisordner berechnet und werden diese Pfade aktualisiert.
OUTPUT_DIR = RESULTS_ROOT
EVALUATION_OUTPUT_DIR = OUTPUT_DIR
OUTPUT_XLSX = EVALUATION_OUTPUT_DIR / OUTPUT_XLSX_FILENAME
OUTPUT_CONFIG_JSON = EVALUATION_OUTPUT_DIR / ANALYSIS_CONFIG_FILENAME


# -----------------------------------------------------------------------------
# Eingelesene Agentendatei und Metriken
# -----------------------------------------------------------------------------
# Für das ursprüngliche Notebook: Data/agent_sc_level_1.csv
AGENT_LEVEL = 1

METRICS = [
    "MAE",
    "MSE",
    "R2",
    "BWR_order",
    "BWR_inventory",
    "IVR",
    "OVR",
]


# -----------------------------------------------------------------------------
# Auswertungsfenster
# -----------------------------------------------------------------------------
# None = cfg["sim"]["testing_time"] verwenden.
# Beispiel für eine feste Fensterlänge:
# TEST_INTERVAL_OVERRIDE = 50
TEST_INTERVAL_OVERRIDE = 50

# None = cfg["sim"]["convergence_time"] +
#        cfg["sim"]["simulation_time"] verwenden.
# Beispiel:
# TEST_START_TIME_OVERRIDE = 5000
TEST_START_TIME_OVERRIDE = None


# -----------------------------------------------------------------------------
# Plot- und Tabellenoptionen
# -----------------------------------------------------------------------------
# Dieses Notebook erzeugt bewusst keine Bilder oder Plots.
SAVE_FIGURES = False

# Die Summary-Tabellen werden in die mehrblättrige Excel-Datei geschrieben.
SAVE_TABLES = True

# Bleibt für Kompatibilität mit der ursprünglichen Plotlogik vorhanden.
# Trainingstypen in dieser Liste wären ausschließlich aus Plots ausgeschlossen,
# nicht aus Tabellen oder statistischen Tests.
PLOT_EXCLUDE_TRAINING_TYPES = []


# -----------------------------------------------------------------------------
# System- und Szenarioebene
# -----------------------------------------------------------------------------
# Zuerst Agentenfehler berechnen, anschließend die Agentenmetriken summieren.
SYSTEM_VIEW_LEVEL = "system_sum_agents"
SCENARIO_TABLE_LEVEL = SYSTEM_VIEW_LEVEL

SCENARIO_TABLE_METRICS = [
    "MAE",
    "MSE",
    "R2",
    "BWR_order",
    "BWR_inventory",
    "IVR",
    "OVR",
]

SCENARIO_TABLE_INCLUDE_ALL_MODEL_COLUMNS = True

# Diese Dimensionen bleiben zusätzlich zur vollständigen Markt-Konfigurations-ID
# als lesbare Szenariometadaten erhalten.
SCENARIO_DIM_COLUMNS = [
    "noise_level",
    "seasonality_frequency",
    "seasonality_magnitude",
]


# -----------------------------------------------------------------------------
# Modellbezeichnungen und Reihenfolge
# -----------------------------------------------------------------------------
MODEL_LABELS = {
    None: "MA / no training",
    "None": "MA / no training",
    "null": "MA / no training",
    "": "MA / no training",
    "local_multichannel": "LSTM local",
    "split_multichannel": "LSTM split",
    "TimesFM_zero_shot": "TimesFM zero-shot",
    "Chronos_zero_shot": "Chronos zero-shot",
    "local_timemixer": "TimeMixer local",
    "split_timemixer": "TimeMixer split",
    "local_patchtst": "PatchTST local",
    "split_patchtst": "PatchTST split",
}

MODEL_ORDER = [
    "MA / no training",
    "Chronos zero-shot",
    "TimesFM zero-shot",
    "LSTM local",
    "LSTM split",
    "TimeMixer local",
    "TimeMixer split",
    "PatchTST local",
    "PatchTST split",
]

LOWER_IS_BETTER = {
    "MAE": True,
    "MSE": True,
    "R2": False,
    "BWR_order": True,
    "BWR_inventory": True,
    "IVR": True,
    "OVR": True,
}


# -----------------------------------------------------------------------------
# Konfiguration prüfen und anzeigen
# -----------------------------------------------------------------------------
if not USE_ZIP_INPUT and not AUTO_DISCOVER_REPORTING_ROOTS:
    missing_roots = [
        Path(root)
        for root in REPORTING_ROOTS
        if not Path(root).exists()
    ]
    if missing_roots:
        raise FileNotFoundError(
            "At least one REPORTING_ROOTS entry does not exist:\n"
            + "\n".join(f" - {root}" for root in missing_roots)
            + "\nSet REPORTING_ROOTS to folders that directly contain "
              "the configuration folders."
        )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("USE_ZIP_INPUT:", USE_ZIP_INPUT)

if USE_ZIP_INPUT:
    print("INPUT_ZIP:", INPUT_ZIP)
    print("EXTRACT_BASE:", EXTRACT_BASE)
else:
    print("synthethic:", synthethic)

print("REPORTING_ROOTS:")
for root in REPORTING_ROOTS:
    print(" -", root, "exists=", Path(root).exists())

print("AUTO_DISCOVER_REPORTING_ROOTS:", AUTO_DISCOVER_REPORTING_ROOTS)
print("REPORTING_BASE:", REPORTING_BASE)
print(
    "Merge same first-level experiment folder:",
    MERGE_SAME_EXPERIMENT_FOLDER_ACROSS_ROOTS,
)
print("Config match strict:", CONFIG_MATCH_STRICT)
print("Duplicate run-id policy:", DUPLICATE_RUN_ID_POLICY)
print("Agent source level:", AGENT_LEVEL)
print("Test interval override:", TEST_INTERVAL_OVERRIDE)
print("Test start override:", TEST_START_TIME_OVERRIDE)
print("Results root directory:", RESULTS_ROOT)
print("Requested results folder name:", RESULTS_FOLDER_NAME or "<automatic>")
print("Output Excel filename:", OUTPUT_XLSX_FILENAME)
print("Analysis config filename:", ANALYSIS_CONFIG_FILENAME)

## 2. Originale Helper-, Config- und Dateierkennungslogik

Die folgende Zelle wurde aus dem ursprünglichen Notebook übernommen.

In [ ]:
def get_cfg_path(cfg: dict, path: list[str], default=np.nan):
    current = cfg
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return default if current is None else current


def first_existing_cfg_path(cfg: dict, paths: list[list[str]], default=np.nan):
    for path in paths:
        value = get_cfg_path(cfg, path, default=None)
        if value is None:
            continue
        if isinstance(value, str) and value.strip() == "":
            continue
        return value
    return default


def find_cfg_value_by_key(cfg: dict, candidate_keys, default=np.nan):
    """Find the first non-empty value whose key matches one of candidate_keys."""
    normalized_candidates = {
        re.sub(r"[^a-z0-9]+", "", str(key).lower())
        for key in candidate_keys
    }
    missing = object()

    def walk(value):
        if isinstance(value, dict):
            # Exact key matches at the current level have priority.
            for key, child in value.items():
                normalized_key = re.sub(r"[^a-z0-9]+", "", str(key).lower())
                if normalized_key in normalized_candidates:
                    if child is not None and not (isinstance(child, str) and child.strip() == ""):
                        return child
            # Then search nested dictionaries/lists.
            for child in value.values():
                found = walk(child)
                if found is not missing:
                    return found
        elif isinstance(value, list):
            for child in value:
                found = walk(child)
                if found is not missing:
                    return found
        return missing

    result = walk(cfg)
    return default if result is missing else result


def normalize_training_type(value):
    if value is None:
        return "None"
    text = str(value).strip()
    if text.lower() in {"", "none", "null", "nan"}:
        return "None"
    return text


def model_label(training_type):
    training_type = normalize_training_type(training_type)
    return MODEL_LABELS.get(training_type, training_type)


def get_lead_values(cfg: dict):
    sc_levels = get_cfg_path(cfg, ["supply_chain", "sc_levels"], default={})
    if not isinstance(sc_levels, dict):
        value = first_existing_cfg_path(
            cfg,
            [["supply_chain", "lead_time"], ["supply_chain", "leadtime"], ["market", "lead_time"]],
            default=np.nan,
        )
        return {"lead_time": value, "lead_time_l0": value, "lead_time_l1": np.nan}

    lead_values = {}
    compact_values = []

    for level_name in sorted(sc_levels.keys()):
        level_cfg = sc_levels[level_name]
        if not isinstance(level_cfg, dict):
            continue

        match = re.search(r"(\d+)$", str(level_name))
        suffix = match.group(1) if match else str(len(lead_values))
        value = level_cfg.get("lead_time", level_cfg.get("leadtime", np.nan))

        lead_values[f"lead_time_l{suffix}"] = value
        compact_values.append(value)

    if not compact_values:
        lead = np.nan
    elif all(v == compact_values[0] for v in compact_values):
        lead = compact_values[0]
    else:
        lead = "-".join(str(v) for v in compact_values)

    lead_values["lead_time"] = lead
    lead_values.setdefault("lead_time_l0", np.nan)
    lead_values.setdefault("lead_time_l1", np.nan)
    return lead_values


def extract_config_metadata(cfg: dict, experiment_folder: str):
    training_type = normalize_training_type(get_cfg_path(cfg, ["sim", "training_type"], default=None))

    lead_values = get_lead_values(cfg)

    demand_split = get_cfg_path(cfg, ["market", "demand_split"], default=None)
    demand_share_0 = demand_split[0] if isinstance(demand_split, list) and len(demand_split) > 0 else np.nan

    lambda_value = first_existing_cfg_path(
        cfg,
        [
            ["market", "lam"],
            ["market", "lambda"],
            ["market", "lambda_value"],
            ["market", "dependency_lambda"],
            ["market", "demand_dependency_lambda"],
            ["market", "shared_variance_lambda"],
            ["lam"],
            ["lambda"],
            ["lambda_value"],
        ],
        default=None,
    )
    if lambda_value is None:
        lambda_value = find_cfg_value_by_key(
            cfg,
            [
                "lam",
                "lambda",
                "lambda_value",
                "dependency_lambda",
                "demand_dependency_lambda",
                "shared_variance_lambda",
                "common_factor_lambda",
            ],
            default=np.nan,
        )

    tau_value = first_existing_cfg_path(
        cfg,
        [
            ["market", "tau"],
            ["market", "tau_value"],
            ["market", "dependency_tau"],
            ["market", "demand_dependency_tau"],
            ["market", "temporal_displacement"],
            ["market", "time_delay"],
            ["tau"],
            ["tau_value"],
        ],
        default=None,
    )
    if tau_value is None:
        tau_value = find_cfg_value_by_key(
            cfg,
            [
                "tau",
                "tau_value",
                "dependency_tau",
                "demand_dependency_tau",
                "temporal_displacement",
                "time_delay",
                "demand_delay",
            ],
            default=np.nan,
        )

    data_source = first_existing_cfg_path(
        cfg,
        [
            ["market", "data_scource"],
            ["market", "data_source"],
            ["market", "source"],
            ["data_source"],
        ],
        default=np.nan,
    )

    meta = {
        "experiment_folder": experiment_folder,
        "training_type": training_type,
        "model": training_type,
        "model_label": model_label(training_type),

        # Wichtig: Schreibweise "seasonality_frequncy" wird unterstützt,
        # weil sie in deiner config so vorkommt.
        "seasonality_frequency": first_existing_cfg_path(
            cfg,
            [
                ["market", "seasonality_frequncy"],
                ["market", "seasonality_frequency"],
                ["market", "frequency"],
            ],
            default=np.nan,
        ),
        "seasonality_magnitude": first_existing_cfg_path(
            cfg,
            [["market", "seasonality_magnitude"], ["market", "seasonality_mag"]],
            default=np.nan,
        ),
        "noise_level": first_existing_cfg_path(
            cfg,
            [
                # In the updated config, random_walk.mean is the active noise scale/std.
                # random_walk.variance is retained there only for compatibility.
                ["market", "random_walk", "mean"],
                ["market", "random_walk", "std"],
                ["market", "random_walk", "sigma"],
                ["market", "noise_level"],
                ["market", "noise"],
                ["market", "random_walk", "variance"],
            ],
            default=np.nan,
        ),
        "random_walk_mean": get_cfg_path(cfg, ["market", "random_walk", "mean"], default=np.nan),
        "demand_share_0": demand_share_0,
        "lambda_value": lambda_value,
        "tau_value": tau_value,
        "data_source": data_source,
        "testing_time": get_cfg_path(cfg, ["sim", "testing_time"], default=np.nan),
        "training_time": get_cfg_path(cfg, ["sim", "training_time"], default=np.nan),
        "simulation_time": get_cfg_path(cfg, ["sim", "simulation_time"], default=np.nan),
        "convergence_time": get_cfg_path(cfg, ["sim", "convergence_time"], default=np.nan),
        "epochs": get_cfg_path(cfg, ["sim", "epochs"], default=np.nan),
        "sequence_length": get_cfg_path(cfg, ["sim", "sequence_length"], default=np.nan),
    }
    meta.update(lead_values)
    return meta


def natural_sort_key(path: Path):
    parts = re.split(r"(\d+)", path.name)
    return [int(p) if p.isdigit() else p.lower() for p in parts]


def find_child_case_insensitive(parent: Path, name: str):
    if not parent.exists():
        return None
    target = name.lower()
    for child in parent.iterdir():
        if child.name.lower() == target:
            return child
    return None


def find_data_dir(run_dir: Path):
    # supports both data/ and Data/
    for candidate in ["data", "Data"]:
        child = find_child_case_insensitive(run_dir, candidate)
        if child is not None and child.is_dir():
            return child
    return None


def find_csv_case_insensitive(parent: Path, filename: str):
    if parent is None or not parent.exists():
        return None
    target = filename.lower()
    for child in parent.iterdir():
        if child.is_file() and child.name.lower() == target:
            return child
    return None


def parse_run_id(run_dir: Path):
    match = re.search(r"run[_-]?(\d+)", run_dir.name.lower())
    return int(match.group(1)) if match else np.nan


# -----------------------------------------------------------------------------
# Config identity / compatibility for merging partial run folders
# -----------------------------------------------------------------------------
# These fields are inferred from the merged factorial config you provided.
# They define the experiment setting and must match before runs from the same
# first-level experiment folder are merged across different reporting roots.
#
# Intentionally excluded:
# - sim.simulation_runs: can describe only the size of a particular partial batch
# - sim.run_start_id: can differ when runs are split across batches
# - transient paths, timestamps, seeds, generated output metadata
CENTRAL_CONFIG_SIGNATURE_SPECS = [
    ("sim.training_type", [["sim", "training_type"]]),
    ("sim.convergence_time", [["sim", "convergence_time"]]),
    ("sim.simulation_time", [["sim", "simulation_time"]]),
    ("sim.training_time", [["sim", "training_time"]]),
    ("sim.testing_time", [["sim", "testing_time"]]),
    ("sim.train_size", [["sim", "train_size"]]),
    ("sim.val_size", [["sim", "val_size"]]),
    ("sim.epochs", [["sim", "epochs"]]),
    ("sim.learning_rate", [["sim", "learning_rate"]]),
    ("sim.momentum", [["sim", "momentum"]]),
    ("sim.batch_size", [["sim", "batch_size"]]),
    ("sim.sequence_length", [["sim", "sequence_length"]]),
    ("early_stopping", [["early_stopping"]]),
    ("market.data_scource", [["market", "data_scource"], ["market", "data_source"]]),
    ("market.primary_demand", [["market", "primary_demand"]]),
    ("market.trend_magnitude", [["market", "trend_magnitude"]]),
    ("market.seasonality_magnitude", [["market", "seasonality_magnitude"]]),
    ("market.seasonality_frequency", [["market", "seasonality_frequncy"], ["market", "seasonality_frequency"], ["market", "frequency"]]),
    ("market.random_walk.mean", [["market", "random_walk", "mean"]]),
    ("market.random_walk.variance", [["market", "random_walk", "variance"], ["market", "noise_level"], ["market", "noise"]]),
    ("market.demand_split", [["market", "demand_split"]]),
    # Whole supply-chain block is central because it includes topology,
    # agents per level, replenishment/forecasting strategy, and lead times.
    ("supply_chain", [["supply_chain"]]),
]


def _is_missing_signature_value(value):
    if value is None:
        return True
    try:
        return bool(pd.isna(value)) and not isinstance(value, (list, tuple, dict, set))
    except Exception:
        return False


def normalize_config_value_for_signature(value):
    """Return a deterministic, JSON-serializable representation for comparisons."""
    if _is_missing_signature_value(value):
        return None
    if isinstance(value, dict):
        return {
            str(k): normalize_config_value_for_signature(v)
            for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))
        }
    if isinstance(value, (list, tuple)):
        return [normalize_config_value_for_signature(v) for v in value]
    if isinstance(value, np.generic):
        return normalize_config_value_for_signature(value.item())
    if isinstance(value, float) and np.isfinite(value):
        # Stable representation for values that may be written as 1 vs 1.0.
        return int(value) if value.is_integer() else value
    return value


def infer_experiment_signature(cfg: dict):
    """
    Build the canonical signature used for merging partial runs.

    Same first-level experiment folders across different reporting roots are
    considered the same experiment only if this signature is identical.
    """
    signature = {}
    for label, paths in CENTRAL_CONFIG_SIGNATURE_SPECS:
        value = first_existing_cfg_path(cfg, paths, default=None)
        if label == "sim.training_type":
            value = normalize_training_type(value)
        signature[label] = normalize_config_value_for_signature(value)
    return signature


def config_signature_hash(signature: dict):
    import hashlib

    payload = json.dumps(signature, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:12]


def diff_config_signatures(reference: dict, candidate: dict):
    diffs = []
    all_keys = sorted(set(reference) | set(candidate))
    for key in all_keys:
        ref_value = reference.get(key, None)
        cand_value = candidate.get(key, None)
        if ref_value != cand_value:
            diffs.append((key, ref_value, cand_value))
    return diffs


def format_config_signature_diffs(diffs, max_items=20):
    lines = []
    for key, ref_value, cand_value in diffs[:max_items]:
        lines.append(f" - {key}: reference={ref_value!r} | candidate={cand_value!r}")
    if len(diffs) > max_items:
        lines.append(f" - ... {len(diffs) - max_items} more differing fields")
    return "\n".join(lines)


def looks_like_reporting_root(path: Path):
    path = Path(path)
    if not path.exists() or not path.is_dir():
        return False
    return any(child.is_dir() and (child / "config.json").exists() for child in path.iterdir())


def discover_reporting_roots_from_base(base=REPORTING_BASE, name_pattern=REPORTING_ROOT_NAME_PATTERN):
    base = Path(base)
    if not base.exists():
        raise FileNotFoundError(f"REPORTING_BASE does not exist: {base}")

    roots = []
    regex = re.compile(name_pattern) if name_pattern else None
    for child in sorted([p for p in base.iterdir() if p.is_dir()], key=natural_sort_key):
        if regex and not regex.search(child.name):
            continue
        if looks_like_reporting_root(child):
            roots.append(child)
    return roots


def resolve_reporting_roots(reporting_roots=None):
    if reporting_roots is None:
        reporting_roots = REPORTING_ROOTS

    roots = []
    if reporting_roots is not None:
        if isinstance(reporting_roots, (str, Path)):
            reporting_roots = [reporting_roots]
        roots.extend(Path(root) for root in reporting_roots)

    if AUTO_DISCOVER_REPORTING_ROOTS:
        roots.extend(discover_reporting_roots_from_base(REPORTING_BASE, REPORTING_ROOT_NAME_PATTERN))

    # De-duplicate while preserving order.
    unique_roots = []
    seen = set()
    for root in roots:
        key = str(root.resolve()) if root.exists() else str(root)
        if key not in seen:
            seen.add(key)
            unique_roots.append(root)

    if not unique_roots:
        raise RuntimeError("No reporting roots configured or discovered.")

    missing = [root for root in unique_roots if not root.exists()]
    if missing:
        raise FileNotFoundError(
            "The following reporting roots do not exist:\n"
            + "\n".join(f" - {root}" for root in missing)
        )

    return unique_roots


## 3. Originale Discovery- und Merge-Logik

Die folgende Zelle entdeckt Experiment-Instanzen und führt partielle Run-Ordner anhand des ersten Experimentordners und der geprüften Config-Signatur zusammen.

In [ ]:
def load_config_json(cfg_path: Path) -> dict:
    """
    Read one experiment config with useful diagnostics.

    Empty, incomplete, malformed, or non-object JSON files raise a ValueError
    that includes the exact file path. UTF-8 BOMs are accepted.
    """
    cfg_path = Path(cfg_path)

    try:
        raw = cfg_path.read_text(encoding="utf-8-sig")
    except OSError as exc:
        raise OSError(f"Config konnte nicht gelesen werden: {cfg_path}: {exc}") from exc

    if not raw.strip():
        raise ValueError(f"Leere config.json wird übersprungen: {cfg_path}")

    try:
        cfg = json.loads(raw)
    except json.JSONDecodeError as exc:
        line = raw.splitlines()[exc.lineno - 1] if raw.splitlines() and exc.lineno <= len(raw.splitlines()) else ""
        pointer = " " * max(exc.colno - 1, 0) + "^"
        raise ValueError(
            "Ungültige config.json wird übersprungen:\n"
            f"  Datei: {cfg_path}\n"
            f"  JSON-Fehler: {exc.msg} (Zeile {exc.lineno}, Spalte {exc.colno})\n"
            f"  Inhalt: {line[:200]}\n"
            f"          {pointer[:200]}"
        ) from exc

    if not isinstance(cfg, dict):
        raise ValueError(
            f"config.json wird übersprungen, weil das oberste JSON-Element kein Objekt ist: "
            f"{cfg_path} (Typ: {type(cfg).__name__})"
        )

    return cfg


def discover_experiment_instances(reporting_roots=None):
    """
    Discover every first-level experiment folder in every reporting root.

    Expected structure for each root:

    REPORTING_ROOT/
      chronos_zero_shot_005/
        config.json
        run_0/
        run_1/
      timesfm_zero_shot_006/
        config.json
        run_0/
        ...
    """

    roots = resolve_reporting_roots(reporting_roots)
    rows = []
    invalid_configs = []

    for reporting_root in roots:
        reporting_root = Path(reporting_root)
        if not looks_like_reporting_root(reporting_root):
            warnings.warn(
                f"Reporting root has no direct experiment folders with config.json: {reporting_root}"
            )
            continue

        exp_dirs = sorted(
            [p for p in reporting_root.iterdir() if p.is_dir()],
            key=natural_sort_key,
        )

        for exp_dir in exp_dirs:
            cfg_path = exp_dir / "config.json"

            if not cfg_path.exists():
                continue

            try:
                cfg = load_config_json(cfg_path)
            except (OSError, ValueError) as exc:
                invalid_configs.append({
                    "config_path": str(cfg_path),
                    "experiment_folder": exp_dir.name,
                    "reporting_root": str(reporting_root),
                    "error": str(exc),
                })
                warnings.warn(str(exc))
                continue

            meta = extract_config_metadata(
                cfg,
                experiment_folder=exp_dir.name,
            )
            signature = infer_experiment_signature(cfg)

            run_dirs = sorted(
                [
                    p for p in exp_dir.iterdir()
                    if p.is_dir() and re.match(r"run[_-]?\d+$", p.name.lower())
                ],
                key=natural_sort_key,
            )

            rows.append({
                "experiment_folder": exp_dir.name,
                "experiment_path": str(exp_dir),
                "reporting_root": str(reporting_root),
                "config_path": str(cfg_path),
                "n_runs": len(run_dirs),
                "run_folders": [p.name for p in run_dirs],
                "run_records": [
                    {
                        "run_id": parse_run_id(p),
                        "run_name": p.name,
                        "run_path": str(p),
                        "source_experiment_path": str(exp_dir),
                        "source_config_path": str(cfg_path),
                        "source_reporting_root": str(reporting_root),
                    }
                    for p in run_dirs
                ],
                "config_signature": signature,
                "config_signature_hash": config_signature_hash(signature),
                **meta,
            })

    df = pd.DataFrame(rows)

    if not df.empty:
        for col in [
            "lead_time",
            "lead_time_l0",
            "lead_time_l1",
            "seasonality_frequency",
            "seasonality_magnitude",
            "noise_level",
        ]:
            if col in df.columns:
                df[f"{col}_num"] = pd.to_numeric(df[col], errors="coerce")

    # Store diagnostics without changing the DataFrame columns.
    df.attrs["invalid_configs"] = invalid_configs

    if invalid_configs:
        warnings.warn(
            f"{len(invalid_configs)} ungültige oder leere config.json-Datei(en) wurden übersprungen. "
            "Details stehen in experiment_instances_df.attrs['invalid_configs']."
        )

    return df


def _deduplicate_run_records(run_records, experiment_folder):
    """Apply DUPLICATE_RUN_ID_POLICY to run records within one merged experiment."""
    policy = DUPLICATE_RUN_ID_POLICY
    if policy not in {"error", "keep_first", "keep_last", "keep_all"}:
        raise ValueError(
            "DUPLICATE_RUN_ID_POLICY must be one of: 'error', 'keep_first', 'keep_last', 'keep_all'"
        )

    sorted_records = sorted(
        run_records,
        key=lambda r: (
            np.inf if pd.isna(r.get("run_id")) else r.get("run_id"),
            str(r.get("source_reporting_root", "")),
            str(r.get("run_path", "")),
        ),
    )

    if policy == "keep_all":
        return sorted_records

    kept_by_id = {}
    duplicates = []

    for record in sorted_records:
        run_id = record.get("run_id")
        key = ("nan", record.get("run_path")) if pd.isna(run_id) else int(run_id)

        if key in kept_by_id:
            duplicates.append((key, kept_by_id[key], record))
            if policy == "keep_last":
                kept_by_id[key] = record
            elif policy == "keep_first":
                continue
            elif policy == "error":
                pass
        else:
            kept_by_id[key] = record

    if duplicates and policy == "error":
        lines = [
            f"Duplicate run_id(s) found while merging experiment folder {experiment_folder!r}.",
            "This usually means the same run was evaluated twice or two partial batches overlap.",
            "Set DUPLICATE_RUN_ID_POLICY to 'keep_first', 'keep_last', or 'keep_all' only if that is intentional.",
        ]
        for key, first, second in duplicates[:20]:
            lines.append(
                f" - run_id={key}:\n"
                f"   first:  {first.get('run_path')}\n"
                f"   second: {second.get('run_path')}"
            )
        if len(duplicates) > 20:
            lines.append(f" - ... {len(duplicates) - 20} more duplicate run ids")
        raise ValueError("\n".join(lines))

    return sorted(kept_by_id.values(), key=lambda r: (np.inf if pd.isna(r.get("run_id")) else r.get("run_id"), str(r.get("run_path", ""))))


def merge_experiment_instances(instances_df):
    """
    Merge same first-level experiment folder across reporting roots.

    Example:
      Reporting/A/chronos_zero_shot_001/run_0, run_1
      Reporting/B/chronos_zero_shot_001/run_2, ..., run_9

    becomes one row with run_records for run_0 ... run_9, provided the central
    config signature is identical in both config.json files.
    """
    if instances_df.empty:
        return instances_df.copy()

    if not MERGE_SAME_EXPERIMENT_FOLDER_ACROSS_ROOTS:
        out = instances_df.copy()
        out["merged_from_n_instances"] = 1
        out["source_experiment_paths"] = out["experiment_path"].map(lambda x: [x])
        out["source_config_paths"] = out["config_path"].map(lambda x: [x])
        out["source_reporting_roots"] = out["reporting_root"].map(lambda x: [x])
        return out

    merged_rows = []

    for experiment_folder, group in instances_df.groupby("experiment_folder", sort=False):
        group = group.sort_values("reporting_root", key=lambda s: s.map(lambda x: natural_sort_key(Path(str(x)))))
        reference = group.iloc[0]
        reference_signature = reference["config_signature"]
        reference_path = reference["config_path"]

        for _, candidate in group.iloc[1:].iterrows():
            diffs = diff_config_signatures(reference_signature, candidate["config_signature"])
            if diffs:
                message = (
                    f"Config mismatch for experiment folder {experiment_folder!r}.\n"
                    f"Reference config: {reference_path}\n"
                    f"Candidate config: {candidate['config_path']}\n"
                    "Differing central fields:\n"
                    + format_config_signature_diffs(diffs)
                )
                if CONFIG_MATCH_STRICT:
                    raise ValueError(message)
                warnings.warn(message)

        all_run_records = []
        for _, row in group.iterrows():
            all_run_records.extend(row["run_records"])

        all_run_records = _deduplicate_run_records(all_run_records, experiment_folder)

        # Use the reference metadata for the merged group. Config compatibility
        # above guarantees central experiment fields match.
        merged = reference.to_dict()
        merged["experiment_path"] = reference["experiment_path"]  # primary path for backward compatibility
        merged["config_path"] = reference["config_path"]          # primary config for backward compatibility
        merged["n_runs"] = len(all_run_records)
        merged["run_folders"] = [record["run_name"] for record in all_run_records]
        merged["run_records"] = all_run_records
        merged["merged_from_n_instances"] = len(group)
        merged["source_experiment_paths"] = list(group["experiment_path"])
        merged["source_config_paths"] = list(group["config_path"])
        merged["source_reporting_roots"] = list(group["reporting_root"])
        merged["n_reporting_roots"] = len(set(group["reporting_root"]))
        merged_rows.append(merged)

    out = pd.DataFrame(merged_rows)

    if not out.empty:
        for col in [
            "lead_time",
            "lead_time_l0",
            "lead_time_l1",
            "seasonality_frequency",
            "seasonality_magnitude",
            "noise_level",
        ]:
            if col in out.columns:
                out[f"{col}_num"] = pd.to_numeric(out[col], errors="coerce")

    return out


experiment_instances_df = discover_experiment_instances()
invalid_configs_df = pd.DataFrame(
    experiment_instances_df.attrs.get("invalid_configs", [])
)
experiments_df = merge_experiment_instances(experiment_instances_df)

print(f"Gefundene Experiment-Instanzen: {len(experiment_instances_df)}")
print(f"Zusammengeführte Experiment-Gruppen: {len(experiments_df)}")
print(f"Genutzte Reporting-Roots: {len(resolve_reporting_roots())}")
for root in resolve_reporting_roots():
    print(" -", root)

if not invalid_configs_df.empty:
    print(f"Übersprungene ungültige Configs: {len(invalid_configs_df)}")
    display(invalid_configs_df)

if not experiment_instances_df.empty:
    display_cols_instances = [
        "reporting_root", "experiment_folder", "n_runs", "run_folders",
        "model_label", "lead_time", "seasonality_frequency", "seasonality_magnitude", "noise_level", "config_signature_hash",
    ]
    display(experiment_instances_df[[c for c in display_cols_instances if c in experiment_instances_df.columns]].head(30))

display_cols_groups = [
    "experiment_folder", "merged_from_n_instances", "n_reporting_roots", "n_runs", "run_folders",
    "model_label", "lead_time", "seasonality_frequency", "seasonality_magnitude", "noise_level", "config_signature_hash",
]
if not experiments_df.empty:
    display(experiments_df[[c for c in display_cols_groups if c in experiments_df.columns]].head(30))

if experiments_df.empty:
    raise RuntimeError(
        "Keine Experiment-Ordner gefunden. Prüfe, ob REPORTING_ROOTS direkt auf die Ordner zeigen, "
        "die die Konfigurationsordner enthalten und ob dort je Ordner eine config.json liegt."
    )

experiments_df.groupby(
    ["lead_time", "model_label"],
    dropna=False,
).size().rename("n_configs").reset_index()


## 4. Originale Metrikberechnung

Je Run wird `Data/agent_sc_level_1.csv` verwendet. Der Systemwert `system_sum_agents` ist die Summe der zuvor je Agent berechneten Metrik.

In [ ]:
def _to_numeric_array(series):
    return pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)


def _safe_ratio(num, den):
    if den is None or np.isnan(den) or den == 0:
        return np.nan
    return num / den


def _cv_ratio(numerator_series, denominator_series):
    numerator_series = np.asarray(numerator_series, dtype=float)
    denominator_series = np.asarray(denominator_series, dtype=float)

    num_mean = np.nanmean(numerator_series)
    den_mean = np.nanmean(denominator_series)
    num_var = np.nanvar(numerator_series)
    den_var = np.nanvar(denominator_series)

    return _safe_ratio(_safe_ratio(num_var, num_mean), _safe_ratio(den_var, den_mean))


def _variance_ratio(numerator_series, denominator_series):
    numerator_series = np.asarray(numerator_series, dtype=float)
    denominator_series = np.asarray(denominator_series, dtype=float)

    num_var = np.nanvar(numerator_series)
    den_var = np.nanvar(denominator_series)

    return _safe_ratio(num_var, den_var)


def _metric_value(metric, demand, forecast=None, order=None, inv=None):
    metric = metric.upper()

    if metric in {"MAE", "MSE", "R2"}:
        if demand is None or forecast is None:
            return np.nan

        demand = np.asarray(demand, dtype=float)
        forecast = np.asarray(forecast, dtype=float)

        mask = np.isfinite(demand) & np.isfinite(forecast)
        demand = demand[mask]
        forecast = forecast[mask]

        if len(demand) < 2:
            return np.nan

        if metric == "MAE":
            return mean_absolute_error(demand, forecast)
        if metric == "MSE":
            return mean_squared_error(demand, forecast)
        if metric == "R2":
            return r2_score(demand, forecast)

    if metric == "BWR_ORDER":
        return _variance_ratio(order, demand)

    if metric == "BWR_INVENTORY":
        return _variance_ratio(inv, demand)

    if metric == "IVR":
        return _cv_ratio(inv, demand)

    if metric == "OVR":
        return _cv_ratio(order, demand)

    raise ValueError(f"Unknown metric: {metric}")


def _normalize_window(window):
    if window is None or pd.isna(window):
        return None
    window = int(window)
    return None if window <= 0 else window


def _first_position_at_or_after_test_start(g: pd.DataFrame, test_start_time):
    """
    Return the first row position of the post-training test window.

    If a time column exists:
      - 0-based time series: first row with time >= test_start_time.
      - 1-based time series: first row with time > test_start_time.

    If no time column exists, test_start_time is interpreted as a 0-based row
    offset from the beginning of that agent's time series.
    """
    if test_start_time is None or pd.isna(test_start_time):
        return None

    test_start_time = float(test_start_time)

    if "time" in g.columns:
        times = pd.to_numeric(g["time"], errors="coerce").to_numpy(dtype=float)
        finite_times = times[np.isfinite(times)]

        if finite_times.size:
            if np.nanmin(finite_times) <= 0:
                mask = times >= test_start_time
            else:
                mask = times > test_start_time

            positions = np.flatnonzero(mask & np.isfinite(times))
            if positions.size:
                return int(positions[0])

            return len(g)

    # Fallback when there is no usable time column.
    return max(0, min(len(g), int(test_start_time)))


def _prepare_agent_arrays(agent_df: pd.DataFrame, window: int, test_start_time=None):
    """
    Return per-agent arrays for the evaluation window.

    If test_start_time is given, the window starts at the first timestep after
    the pre-training simulation and then keeps the next `window` observations.
    This is the desired post-update testing window, not the last values of the
    whole time series.

    For forecasting metrics, forecast(t-1) is compared with demand(t). Therefore
    the forecast-aligned arrays include the row immediately before the test
    window when it is available, so the first post-update demand can be scored.
    """
    df = agent_df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]

    required = {"id", "demand"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in agent file: {missing}")

    if "time" in df.columns:
        df = df.sort_values(["id", "time"])

    window = _normalize_window(window)
    result = {}

    for agent_id, g in df.groupby("id", sort=True):
        g = g.reset_index(drop=True)

        start_pos = _first_position_at_or_after_test_start(g, test_start_time)

        if start_pos is None:
            # Backward-compatible fallback: if no test start is available, use
            # the last `window` rows as before.
            start_pos = max(0, len(g) - window) if window is not None else 0
            include_previous_forecast_row = False
        else:
            include_previous_forecast_row = True

        end_pos = len(g) if window is None else min(len(g), start_pos + window)

        g_test = g.iloc[start_pos:end_pos]

        if include_previous_forecast_row:
            # Include the row right before the test window so forecast(t-1) can
            # be compared with the first demand inside the test window.
            forecast_start_pos = max(0, start_pos - 1)
            g_forecast_window = g.iloc[forecast_start_pos:end_pos]
        else:
            # Old behavior for fallback mode.
            g_forecast_window = g_test

        demand_raw = _to_numeric_array(g_test["demand"])

        values = {
            "demand_raw": demand_raw,
            "demand_forecast_aligned": (
                _to_numeric_array(g_forecast_window["demand"])[1:]
                if len(g_forecast_window) >= 2
                else np.array([])
            ),
        }

        if "forecast" in g.columns:
            forecast_raw = _to_numeric_array(g_test["forecast"])
            values["forecast_raw"] = forecast_raw
            values["forecast_aligned"] = (
                _to_numeric_array(g_forecast_window["forecast"])[:-1]
                if len(g_forecast_window) >= 2
                else np.array([])
            )
        else:
            values["forecast_raw"] = np.array([])
            values["forecast_aligned"] = np.array([])

        if "order" in g.columns:
            values["order_raw"] = _to_numeric_array(g_test["order"])
        else:
            values["order_raw"] = np.full_like(demand_raw, np.nan)

        if "inv" in g.columns:
            values["inv_raw"] = _to_numeric_array(g_test["inv"])
        elif "inventory" in g.columns:
            values["inv_raw"] = _to_numeric_array(g_test["inventory"])
        else:
            values["inv_raw"] = np.full_like(demand_raw, np.nan)

        values["test_start_position"] = start_pos
        values["test_end_position_exclusive"] = end_pos
        result[agent_id] = values

    return result


def _stack_min_length(arrays):
    arrays = [np.asarray(a, dtype=float) for a in arrays if len(a) > 0]
    if not arrays:
        return np.array([])
    min_len = min(len(a) for a in arrays)
    if min_len <= 0:
        return np.array([])
    return np.vstack([a[-min_len:] for a in arrays])


def compute_metrics_from_agent_df(
    agent_df: pd.DataFrame,
    metrics=METRICS,
    test_interval=50,
    test_start_time=None,
):
    agent_arrays = _prepare_agent_arrays(
        agent_df,
        window=test_interval,
        test_start_time=test_start_time,
    )

    rows = []

    for metric in metrics:
        metric_key = metric.upper()

        per_agent_values = []

        for agent_id, values in agent_arrays.items():
            if metric_key in {"MAE", "MSE", "R2"}:
                value = _metric_value(
                    metric_key,
                    demand=values["demand_forecast_aligned"],
                    forecast=values["forecast_aligned"],
                )
            elif metric_key == "BWR_ORDER":
                value = _metric_value(metric_key, demand=values["demand_raw"], order=values["order_raw"])
            elif metric_key == "BWR_INVENTORY":
                value = _metric_value(metric_key, demand=values["demand_raw"], inv=values["inv_raw"])
            elif metric_key == "IVR":
                value = _metric_value(metric_key, demand=values["demand_raw"], inv=values["inv_raw"])
            elif metric_key == "OVR":
                value = _metric_value(metric_key, demand=values["demand_raw"], order=values["order_raw"])
            else:
                raise ValueError(metric)

            per_agent_values.append(value)

            rows.append({
                "metric": metric,
                "level": "agent",
                "agent_id": agent_id,
                "value": value,
                "test_start_position": values.get("test_start_position", np.nan),
                "test_end_position_exclusive": values.get("test_end_position_exclusive", np.nan),
            })

        # Summe der Agentenmetriken wie im alten Notebook.
        rows.append({
            "metric": metric,
            "level": "system_sum_agents",
            "agent_id": np.nan,
            "value": np.nansum(per_agent_values) if len(per_agent_values) else np.nan,
        })

        # Aggregierte Systemmetrik: alle Agenten pro Zeitschritt summieren.
        if metric_key in {"MAE", "MSE", "R2"}:
            demand_stack = _stack_min_length([v["demand_forecast_aligned"] for v in agent_arrays.values()])
            forecast_stack = _stack_min_length([v["forecast_aligned"] for v in agent_arrays.values()])
            if demand_stack.size and forecast_stack.size:
                demand_agg = np.nansum(demand_stack, axis=0)
                forecast_agg = np.nansum(forecast_stack, axis=0)
                agg_value = _metric_value(metric_key, demand=demand_agg, forecast=forecast_agg)
            else:
                agg_value = np.nan

        elif metric_key == "BWR_ORDER":
            demand_stack = _stack_min_length([v["demand_raw"] for v in agent_arrays.values()])
            order_stack = _stack_min_length([v["order_raw"] for v in agent_arrays.values()])
            agg_value = _metric_value(
                metric_key,
                demand=np.nansum(demand_stack, axis=0),
                order=np.nansum(order_stack, axis=0),
            ) if demand_stack.size and order_stack.size else np.nan

        elif metric_key == "BWR_INVENTORY":
            demand_stack = _stack_min_length([v["demand_raw"] for v in agent_arrays.values()])
            inv_stack = _stack_min_length([v["inv_raw"] for v in agent_arrays.values()])
            agg_value = _metric_value(
                metric_key,
                demand=np.nansum(demand_stack, axis=0),
                inv=np.nansum(inv_stack, axis=0),
            ) if demand_stack.size and inv_stack.size else np.nan

        elif metric_key == "IVR":
            demand_stack = _stack_min_length([v["demand_raw"] for v in agent_arrays.values()])
            inv_stack = _stack_min_length([v["inv_raw"] for v in agent_arrays.values()])
            agg_value = _metric_value(
                metric_key,
                demand=np.nansum(demand_stack, axis=0),
                inv=np.nansum(inv_stack, axis=0),
            ) if demand_stack.size and inv_stack.size else np.nan

        elif metric_key == "OVR":
            demand_stack = _stack_min_length([v["demand_raw"] for v in agent_arrays.values()])
            order_stack = _stack_min_length([v["order_raw"] for v in agent_arrays.values()])
            agg_value = _metric_value(
                metric_key,
                demand=np.nansum(demand_stack, axis=0),
                order=np.nansum(order_stack, axis=0),
            ) if demand_stack.size and order_stack.size else np.nan

        rows.append({
            "metric": metric,
            "level": "system_aggregated",
            "agent_id": np.nan,
            "value": agg_value,
        })

    return pd.DataFrame(rows)


## 5. Originales Einlesen und Mergen der Runs zu `run_metrics`

Auch `iter_run_records_from_experiment_row()`, `evaluate_experiment()` und `evaluate_all_experiments()` stammen aus dem alten Notebook.

In [ ]:
def iter_run_records_from_experiment_row(exp_row):
    """Return run records for merged or legacy experiment rows."""
    records = exp_row.get("run_records", None)
    if isinstance(records, list) and records:
        return records

    # Fallback for older experiments_df objects that only have experiment_path.
    exp_path = Path(exp_row["experiment_path"])
    run_dirs = sorted(
        [p for p in exp_path.iterdir() if p.is_dir() and re.match(r"run[_-]?\d+", p.name.lower())],
        key=natural_sort_key,
    )
    return [
        {
            "run_id": parse_run_id(p),
            "run_name": p.name,
            "run_path": str(p),
            "source_experiment_path": str(exp_path),
            "source_config_path": str(exp_row.get("config_path", "")),
            "source_reporting_root": str(exp_path.parent),
        }
        for p in run_dirs
    ]


def evaluate_experiment(exp_row):
    cfg_path = Path(exp_row["config_path"])

    with cfg_path.open("r", encoding="utf-8") as f:
        cfg = json.load(f)

    meta = extract_config_metadata(cfg, experiment_folder=exp_row["experiment_folder"])
    testing_time = meta.get("testing_time", np.nan)
    convergence_time = meta.get("convergence_time", np.nan)
    simulation_time = meta.get("simulation_time", np.nan)

    if TEST_INTERVAL_OVERRIDE is None:
        test_interval = int(testing_time) if pd.notna(testing_time) else 50
    else:
        test_interval = int(TEST_INTERVAL_OVERRIDE)

    if TEST_START_TIME_OVERRIDE is None:
        if pd.notna(convergence_time) and pd.notna(simulation_time):
            # Start directly after the pre-training simulation.
            # With your config this is convergence_time + simulation_time.
            test_start_time = int(convergence_time) + int(simulation_time)
        else:
            test_start_time = np.nan
    else:
        test_start_time = int(TEST_START_TIME_OVERRIDE)

    run_records = iter_run_records_from_experiment_row(exp_row)
    all_rows = []

    for record in run_records:
        run_dir = Path(record["run_path"])
        run_id = record.get("run_id", parse_run_id(run_dir))
        data_dir = find_data_dir(run_dir)

        if data_dir is None:
            warnings.warn(f"No data/Data folder found in {run_dir}")
            continue

        agent_file = find_csv_case_insensitive(data_dir, f"agent_sc_level_{AGENT_LEVEL}.csv")
        if agent_file is None:
            warnings.warn(f"No agent_sc_level_{AGENT_LEVEL}.csv found in {data_dir}")
            continue

        try:
            agent_df = pd.read_csv(agent_file)
            metric_df = compute_metrics_from_agent_df(
                agent_df,
                metrics=METRICS,
                test_interval=test_interval,
                test_start_time=test_start_time,
            )
        except Exception as exc:
            warnings.warn(f"Could not evaluate {agent_file}: {exc}")
            continue

        for key, value in meta.items():
            metric_df[key] = value

        metric_df["run_id"] = run_id
        metric_df["run_name"] = record.get("run_name", run_dir.name)
        metric_df["run_path"] = str(run_dir)
        metric_df["agent_file"] = str(agent_file)
        metric_df["test_interval_used"] = test_interval
        metric_df["test_start_time_used"] = test_start_time
        metric_df["test_window_end_time_used"] = (
            test_start_time + test_interval
            if pd.notna(test_start_time) and pd.notna(test_interval)
            else np.nan
        )

        # Provenance for merged runs across multiple reporting roots.
        metric_df["source_reporting_root"] = record.get("source_reporting_root", str(run_dir.parent.parent))
        metric_df["source_experiment_path"] = record.get("source_experiment_path", str(run_dir.parent))
        metric_df["source_config_path"] = record.get("source_config_path", str(cfg_path))
        metric_df["merged_from_n_instances"] = exp_row.get("merged_from_n_instances", 1)
        metric_df["n_reporting_roots"] = exp_row.get("n_reporting_roots", 1)
        metric_df["config_signature_hash"] = exp_row.get("config_signature_hash", np.nan)

        # Convenient scenario label
        metric_df["scenario"] = (
            "noise=" + metric_df["noise_level"].astype(str)
            + " | freq=" + metric_df["seasonality_frequency"].astype(str)
            + " | seasonal_mag=" + metric_df["seasonality_magnitude"].astype(str)
        )

        all_rows.append(metric_df)

    if not all_rows:
        return pd.DataFrame()

    return pd.concat(all_rows, ignore_index=True)


def evaluate_all_experiments(experiments_df):
    result = []
    for _, exp_row in experiments_df.iterrows():
        result.append(evaluate_experiment(exp_row))
    result = [df for df in result if not df.empty]
    if not result:
        return pd.DataFrame()
    return pd.concat(result, ignore_index=True)


run_metrics = evaluate_all_experiments(experiments_df)

print(f"Berechnete Metrik-Zeilen: {len(run_metrics)}")
display(run_metrics.head(20))

if run_metrics.empty:
    raise RuntimeError(
        f"Keine Run-Metriken berechnet. Prüfe, ob run_*/data/agent_sc_level_{AGENT_LEVEL}.csv existiert "
        "oder ob AGENT_LEVEL angepasst werden muss."
    )


def _tag_value(value):
    """Make one value safe and compact for folder/file names."""
    if pd.isna(value):
        return "unknown"

    try:
        value_float = float(value)
        if value_float.is_integer():
            return str(int(value_float))
    except Exception:
        pass

    value_text = str(value).strip()
    value_text = re.sub(r"[^A-Za-z0-9._-]+", "_", value_text)
    return value_text or "unknown"


def _unique_tag_part(df, column, label):
    """Return e.g. start_5000, len_50, or start_mixed."""
    if column not in df.columns:
        return f"{label}_unknown"

    values = df[column].dropna().drop_duplicates().tolist()

    if not values:
        return f"{label}_unknown"

    if len(values) == 1:
        return f"{label}_{_tag_value(values[0])}"

    return f"{label}_mixed"


def build_evaluation_window_tag(metrics_df):
    """
    Build a stable tag for the actual metric-evaluation window.

    The tag is retained in analysis_config.json for traceability, e.g.:
        eval_start_5000_len_50_end_5050
        eval_start_5000_len_100_end_5100
    """
    return "eval_" + "_".join(
        [
            _unique_tag_part(metrics_df, "test_start_time_used", "start"),
            _unique_tag_part(metrics_df, "test_interval_used", "len"),
            _unique_tag_part(metrics_df, "test_window_end_time_used", "end"),
        ]
    )


def _unique_non_missing_values(df: pd.DataFrame, column: str):
    if column not in df.columns:
        return []

    values = []
    for value in df[column].tolist():
        if value is None:
            continue
        try:
            if pd.isna(value):
                continue
        except Exception:
            pass
        if value not in values:
            values.append(value)
    return sorted(values, key=lambda value: str(value))


def _values_folder_tag(df: pd.DataFrame, column: str):
    values = _unique_non_missing_values(df, column)
    if not values:
        return "unknown"
    return "-".join(_tag_value(value) for value in values)


def _sanitize_results_folder_name(name: str):
    text = str(name).strip()
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    text = text.strip("._-")
    if not text:
        raise ValueError("RESULTS_FOLDER_NAME darf nicht nur aus Sonderzeichen bestehen.")
    return text


def _infer_synthetic_flag(experiments: pd.DataFrame):
    # Im manuellen Modus ist die zentrale Auswahl die verlässlichste Quelle.
    if not USE_ZIP_INPUT:
        return bool(synthethic)

    source_values = _unique_non_missing_values(experiments, "data_source")
    source_text = " ".join(str(value).lower() for value in source_values)
    if "synt" in source_text:
        return True
    if "real" in source_text or "empir" in source_text:
        return False

    path_text = " ".join(
        str(value).lower()
        for column in ["reporting_root", "experiment_path", "config_path"]
        for value in _unique_non_missing_values(experiments, column)
    )
    if "synt" in path_text:
        return True
    if "real_world" in path_text or "real-world" in path_text:
        return False

    # Rückwärtskompatibler Fallback, falls die ZIP-Configs keine Datenart tragen.
    return bool(synthethic)


EVALUATION_WINDOW_TAG = build_evaluation_window_tag(run_metrics)
DATA_IS_SYNTHETIC = _infer_synthetic_flag(experiments_df)
LAMBDA_FOLDER_TAG = _values_folder_tag(experiments_df, "lambda_value")
TAU_FOLDER_TAG = _values_folder_tag(experiments_df, "tau_value")

if LAMBDA_FOLDER_TAG == "unknown":
    warnings.warn(
        "Kein Lambda-Wert wurde in den analysierten config.json-Dateien gefunden. "
        "Der automatische Ergebnisordner verwendet deshalb lambda_unknown."
    )
if TAU_FOLDER_TAG == "unknown":
    warnings.warn(
        "Kein Tau-Wert wurde in den analysierten config.json-Dateien gefunden. "
        "Der automatische Ergebnisordner verwendet deshalb tau_unknown."
    )

if RESULTS_FOLDER_NAME is None or str(RESULTS_FOLDER_NAME).strip() == "":
    RESULTS_FOLDER_NAME_USED = (
        f"Results_synth_{str(DATA_IS_SYNTHETIC).lower()}_"
        f"lambda_{LAMBDA_FOLDER_TAG}_tau_{TAU_FOLDER_TAG}"
    )
else:
    RESULTS_FOLDER_NAME_USED = _sanitize_results_folder_name(RESULTS_FOLDER_NAME)

EVALUATION_OUTPUT_DIR = RESULTS_ROOT / RESULTS_FOLDER_NAME_USED
EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Alle Ergebnisdateien dieser Analyse werden gemeinsam in diesem Ordner abgelegt.
OUTPUT_DIR = EVALUATION_OUTPUT_DIR
OUTPUT_XLSX = EVALUATION_OUTPUT_DIR / OUTPUT_XLSX_FILENAME
OUTPUT_CONFIG_JSON = EVALUATION_OUTPUT_DIR / ANALYSIS_CONFIG_FILENAME

print(f"Evaluation window tag: {EVALUATION_WINDOW_TAG}")
print(f"Synthetic data: {DATA_IS_SYNTHETIC}")
print(f"Lambda value(s) from configs: {_unique_non_missing_values(experiments_df, 'lambda_value')}")
print(f"Tau value(s) from configs: {_unique_non_missing_values(experiments_df, 'tau_value')}")
print(f"Export directory for this evaluation: {EVALUATION_OUTPUT_DIR}")
print(f"Summary Excel path: {OUTPUT_XLSX}")
print(f"Analysis config path: {OUTPUT_CONFIG_JSON}")

run_metrics.groupby(["metric", "level", "lead_time", "model_label"], dropna=False)["value"].count().rename("n").reset_index().head(30)


## 6. Neue Auswertungsschicht je Marktkonfiguration

Die Run-ID ist hier nur der Pairing-Schlüssel innerhalb einer Marktkonfiguration. Gezählt werden anschließend Markt­konfigurationen.

In [ ]:
import hashlib
import json
import math
import re
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
from scipy.stats import t as student_t

PRIMARY_METRIC = "MAE"
SYSTEM_LEVEL = "system_sum_agents"
EXPECTED_RUN_IDS = tuple(range(10))
ALPHA = 0.05
MIN_PAIRED_RUNS = 2
FAMILY_PAIRS = {
    "LSTM": ("LSTM local", "LSTM split"),
    "TimeMixer": ("TimeMixer local", "TimeMixer split"),
    "PatchTST": ("PatchTST local", "PatchTST split"),
}


def model_metadata(model: str) -> tuple[str, str]:
    model = str(model)
    if model.startswith("LSTM "):
        return "LSTM", "split" if model.endswith("split") else "local"
    if model.startswith("TimeMixer "):
        return "TimeMixer", "split" if model.endswith("split") else "local"
    if model.startswith("PatchTST "):
        return "PatchTST", "split" if model.endswith("split") else "local"
    if model == "Chronos zero-shot":
        return "Chronos", "zero-shot"
    if model == "TimesFM zero-shot":
        return "TimesFM", "zero-shot"
    if model == "MA / no training":
        return "MA baseline", "baseline"
    return model, "other"


def _normalize_json(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): _normalize_json(v) for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))}
    if isinstance(value, (list, tuple)):
        return [_normalize_json(v) for v in value]
    if isinstance(value, np.generic):
        return _normalize_json(value.item())
    if isinstance(value, float):
        if not np.isfinite(value):
            return None
        return int(value) if value.is_integer() else value
    return value


def stable_hash(payload: dict, length: int = 12) -> str:
    text = json.dumps(_normalize_json(payload), sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:length]


def scenario_payload(cfg: dict) -> dict:
    """Nur die exogenen Faktoren des Marktszenarios."""

    meta = extract_config_metadata(
        cfg,
        experiment_folder="",
    )

    market = (
        cfg.get("market", {})
        if isinstance(cfg.get("market", {}), dict)
        else {}
    )

    return {
        "num_products": market.get("num_products"),
        "lead_time": meta.get("lead_time"),
        "seasonality_frequency": meta.get("seasonality_frequency"),
        "seasonality_magnitude": meta.get("seasonality_magnitude"),
        "noise_level": meta.get("noise_level"),
    }


def _config_scenario_metadata(cfg: dict, lead_time: Any) -> dict:
    market = cfg.get("market", {}) if isinstance(cfg.get("market", {}), dict) else {}
    supply_chain = cfg.get("supply_chain", {}) if isinstance(cfg.get("supply_chain", {}), dict) else {}
    scenario_id = stable_hash(scenario_payload(cfg))
    agents_per_level = supply_chain.get("agents_per_level", [])
    expected_agent_count = (
        agents_per_level[AGENT_LEVEL]
        if isinstance(agents_per_level, list) and len(agents_per_level) > AGENT_LEVEL
        else np.nan
    )
    return {
        "scenario_id": scenario_id,
        "scenario_label": f"LT={lead_time} | market={scenario_id}",
        "num_products": market.get("num_products", np.nan),
        "expected_agent_count": expected_agent_count,
    }


def enrich_old_pipeline_outputs(experiments_df: pd.DataFrame, run_metrics: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Add market-configuration and model-family metadata after the original pipeline has finished."""
    experiments = experiments_df.copy()
    scenario_rows = []
    for _, row in experiments.iterrows():
        cfg_path = Path(row["config_path"])
        with cfg_path.open("r", encoding="utf-8") as f:
            cfg = json.load(f)
        meta = _config_scenario_metadata(cfg, row.get("lead_time", np.nan))
        family, mode = model_metadata(row.get("model_label"))
        records = row.get("run_records", [])
        run_ids = sorted({int(r["run_id"]) for r in records if pd.notna(r.get("run_id"))}) if isinstance(records, list) else []
        scenario_rows.append({
            "config_signature_hash": row.get("config_signature_hash"),
            **meta,
            "model_family": family,
            "learning_mode": mode,
            "run_ids_available": ", ".join(map(str, run_ids)),
            "run_ids_missing": ", ".join(map(str, sorted(set(EXPECTED_RUN_IDS) - set(run_ids)))),
            "n_runs_available": len(run_ids),
            "complete_expected_runs": set(run_ids) == set(EXPECTED_RUN_IDS),
        })
    scenario_meta = pd.DataFrame(scenario_rows)
    experiments = experiments.merge(scenario_meta, on="config_signature_hash", how="left", validate="one_to_one")
    metrics = run_metrics.merge(
        scenario_meta[[
            "config_signature_hash", "scenario_id", "scenario_label", "num_products",
            "expected_agent_count", "model_family", "learning_mode",
        ]],
        on="config_signature_hash",
        how="left",
        validate="many_to_one",
    )
    return experiments, metrics


def summarize_run_metrics(run_metrics: pd.DataFrame) -> pd.DataFrame:
    keys = [
        "scenario_id", "scenario_label", "lead_time", "num_products",
        "seasonality_frequency", "seasonality_magnitude", "noise_level",
        "model_label", "model_family", "learning_mode", "metric", "level", "agent_id",
    ]
    return (
        run_metrics.groupby(keys, dropna=False)["value"]
        .agg(n_runs="count", mean="mean", std="std", median="median", minimum="min", maximum="max")
        .reset_index()
    )


def holm_adjust(pvalues: Iterable[float]) -> np.ndarray:
    p = np.asarray(list(pvalues), dtype=float)
    out = np.full(len(p), np.nan)
    valid_idx = np.flatnonzero(np.isfinite(p))
    if not len(valid_idx):
        return out
    ordered_idx = valid_idx[np.argsort(p[valid_idx])]
    m = len(ordered_idx)
    running = 0.0
    for rank, idx in enumerate(ordered_idx):
        adjusted = min(1.0, (m - rank) * p[idx])
        running = max(running, adjusted)
        out[idx] = running
    return out


def paired_one_sided(a: pd.Series, b: pd.Series, lower_is_better: bool, min_n: int = MIN_PAIRED_RUNS) -> dict:
    paired = pd.concat([a.rename("a"), b.rename("b")], axis=1).dropna()
    diff = paired["a"] - paired["b"]
    n = len(diff)
    mean_diff = float(diff.mean()) if n else np.nan
    sd = float(diff.std(ddof=1)) if n >= 2 else np.nan
    paired_ids = ", ".join(map(str, sorted(pd.to_numeric(paired.index, errors="coerce").dropna().astype(int).tolist())))
    if n < min_n:
        return {
            "paired_run_ids": paired_ids, "n_pairs": n, "mean_diff": mean_diff,
            "diff_sd": sd, "t_stat": np.nan, "p_one_sided": np.nan,
        }
    if sd == 0 or not np.isfinite(sd):
        if mean_diff == 0:
            t_stat, p = 0.0, 0.5
        elif lower_is_better:
            t_stat, p = (-np.inf, 0.0) if mean_diff < 0 else (np.inf, 1.0)
        else:
            t_stat, p = (np.inf, 0.0) if mean_diff > 0 else (-np.inf, 1.0)
    else:
        t_stat = mean_diff / (sd / math.sqrt(n))
        p = float(student_t.cdf(t_stat, df=n - 1)) if lower_is_better else float(student_t.sf(t_stat, df=n - 1))
    return {
        "paired_run_ids": paired_ids, "n_pairs": n, "mean_diff": mean_diff,
        "diff_sd": sd, "t_stat": float(t_stat), "p_one_sided": float(p),
    }


def ordered_models(columns: Iterable[str]) -> list[str]:
    cols = list(columns)
    known = [m for m in MODEL_ORDER if m in cols]
    return known + sorted(m for m in cols if m not in known)


def classify_set(members: list[str], separated_exists: bool = True) -> str:
    if not separated_exists or not members:
        return "no_separated_set"
    modes = [model_metadata(m)[1] for m in members]
    n_split = sum(mode == "split" for mode in modes)
    n_non_split = len(members) - n_split
    if n_split and not n_non_split:
        return "split_only_singleton" if n_split == 1 else "split_only_multiple"
    if n_split and n_non_split:
        return "mixed_with_split"
    return "non_split_only"


def scenario_system_analysis(run_metrics: pd.DataFrame, metric: str = PRIMARY_METRIC, level: str = SYSTEM_LEVEL) -> dict[str, pd.DataFrame]:
    d = run_metrics[
        run_metrics["metric"].eq(metric)
        & run_metrics["level"].eq(level)
        & run_metrics["value"].notna()
        & np.isfinite(run_metrics["value"])
    ].copy()
    scenario_rows, bva_rows, pair_rows, set_member_rows, model_mean_rows = [], [], [], [], []
    scenario_cols = [
        "scenario_id", "scenario_label", "lead_time", "num_products",
        "seasonality_frequency", "seasonality_magnitude", "noise_level",
    ]
    for scenario_id, group in d.groupby("scenario_id", dropna=False):
        scenario = {c: group.iloc[0][c] for c in scenario_cols if c in group.columns}
        pivot = group.pivot_table(index="run_id", columns="model_label", values="value", aggfunc="mean")
        models = ordered_models(pivot.columns)
        means = pivot[models].mean(skipna=True)
        means = means[np.isfinite(means)]
        if means.empty:
            continue
        lower = LOWER_IS_BETTER.get(metric, True)
        ranked = list(means.sort_values(ascending=lower).index)
        best = ranked[0]

        for rank, model in enumerate(ranked, start=1):
            family, mode = model_metadata(model)
            run_ids = sorted(pivot[model].dropna().index.astype(int).tolist())
            model_mean_rows.append({
                **scenario, "metric": metric, "model_label": model, "model_family": family,
                "learning_mode": mode, "rank_by_mean": rank, "mean_system_metric": float(means[model]),
                "n_runs": len(run_ids), "run_ids": ", ".join(map(str, run_ids)),
                "complete_expected_runs": set(run_ids) == set(EXPECTED_RUN_IDS),
            })

        local_bva = []
        for competitor in ranked[1:]:
            test = paired_one_sided(pivot[best], pivot[competitor], lower_is_better=lower)
            paired_ids = [int(x) for x in test["paired_run_ids"].split(", ") if x != ""]
            best_paired_mean = float(pivot.loc[paired_ids, best].mean()) if paired_ids else np.nan
            competitor_paired_mean = float(pivot.loc[paired_ids, competitor].mean()) if paired_ids else np.nan
            local_bva.append({
                **scenario, "metric": metric, "best_model": best, "competitor_model": competitor,
                "best_mean_all_available_runs": float(means[best]),
                "competitor_mean_all_available_runs": float(means[competitor]),
                "best_mean_paired_runs": best_paired_mean,
                "competitor_mean_paired_runs": competitor_paired_mean,
                **test,
            })
        adj = holm_adjust(r["p_one_sided"] for r in local_bva)
        for row, p_adj in zip(local_bva, adj):
            row["p_adj_holm"] = p_adj
            row["test_available"] = bool(np.isfinite(row["p_one_sided"]))
            row["significant_after_holm"] = bool(np.isfinite(p_adj) and p_adj <= ALPHA)
            bva_rows.append(row)

        strict = bool(local_bva) and all(r["test_available"] and r["significant_after_holm"] for r in local_bva)
        best_set = [best] + [r["competitor_model"] for r in local_bva if not r["significant_after_holm"]]
        best_set = [m for m in ranked if m in set(best_set)]

        local_pairs = []
        for i, model_a in enumerate(models):
            for model_b in models[i + 1:]:
                mean_a = float(means.get(model_a, np.nan))
                mean_b = float(means.get(model_b, np.nan))
                if not (np.isfinite(mean_a) and np.isfinite(mean_b)) or mean_a == mean_b:
                    better, worse = None, None
                    test = {"paired_run_ids": "", "n_pairs": 0, "mean_diff": np.nan, "diff_sd": np.nan, "t_stat": np.nan, "p_one_sided": np.nan}
                else:
                    if lower:
                        better, worse = (model_a, model_b) if mean_a < mean_b else (model_b, model_a)
                    else:
                        better, worse = (model_a, model_b) if mean_a > mean_b else (model_b, model_a)
                    test = paired_one_sided(pivot[better], pivot[worse], lower_is_better=lower)
                local_pairs.append({
                    **scenario, "metric": metric, "model_a": model_a, "model_b": model_b,
                    "mean_a_all_available_runs": mean_a, "mean_b_all_available_runs": mean_b,
                    "descriptive_better_model": better, "descriptive_worse_model": worse, **test,
                })
        pair_adj = holm_adjust(r["p_one_sided"] for r in local_pairs)
        for row, p_adj in zip(local_pairs, pair_adj):
            row["p_adj_holm"] = p_adj
            row["test_available"] = bool(np.isfinite(row["p_one_sided"]))
            row["significant_after_holm"] = bool(np.isfinite(p_adj) and p_adj <= ALPHA)
            pair_rows.append(row)

        pair_lookup = {}
        for row in local_pairs:
            if row["descriptive_better_model"] and row["descriptive_worse_model"]:
                pair_lookup[(row["descriptive_better_model"], row["descriptive_worse_model"])] = row

        separated_members, separated_exists = [], False
        for k in range(1, len(ranked) + 1):
            inside, outside = ranked[:k], ranked[k:]
            internal_valid_and_nonsig = True
            for i, a in enumerate(inside):
                for b in inside[i + 1:]:
                    better = a if (means[a] < means[b] if lower else means[a] > means[b]) else b
                    worse = b if better == a else a
                    row = pair_lookup.get((better, worse))
                    if row is None or not row["test_available"] or row["significant_after_holm"]:
                        internal_valid_and_nonsig = False
                        break
                if not internal_valid_and_nonsig:
                    break
            if not internal_valid_and_nonsig:
                continue
            if not outside:
                boundary_ok = True
            else:
                row = pair_lookup.get((inside[-1], outside[0]))
                boundary_ok = bool(row and row["test_available"] and row["significant_after_holm"])
            if boundary_ok:
                separated_members, separated_exists = inside, True
                break

        best_comp = classify_set(best_set, separated_exists=True)
        separated_comp = classify_set(separated_members, separated_exists=separated_exists)
        best_family, best_mode = model_metadata(best)
        best_has_untestable = any(not r["test_available"] for r in local_bva)
        run_sets = [set(pivot[m].dropna().index.astype(int)) for m in ranked]
        common_all = sorted(set.intersection(*run_sets)) if run_sets else []
        scenario_rows.append({
            **scenario, "metric": metric, "n_models": len(ranked), "models_available": ", ".join(ranked),
            "common_run_ids_all_models": ", ".join(map(str, common_all)),
            "n_common_runs_all_models": len(common_all),
            "all_models_have_expected_runs": all(set(pivot[m].dropna().index.astype(int)) == set(EXPECTED_RUN_IDS) for m in ranked),
            "descriptive_best_model": best, "descriptive_best_family": best_family,
            "descriptive_best_learning_mode": best_mode, "descriptive_best_mean": float(means[best]),
            "strict_statistical_winner": strict, "strict_winner_model": best if strict else None,
            "strict_winner_family": best_family if strict else None,
            "statistical_best_set_models": ", ".join(best_set), "statistical_best_set_size": len(best_set),
            "statistical_best_set_n_split": sum(model_metadata(m)[1] == "split" for m in best_set),
            "statistical_best_set_n_non_split": sum(model_metadata(m)[1] != "split" for m in best_set),
            "statistical_best_set_composition": best_comp,
            "best_set_has_untestable_comparisons": best_has_untestable,
            "separated_winning_set_exists": separated_exists,
            "separated_winning_set_models": ", ".join(separated_members),
            "separated_winning_set_size": len(separated_members),
            "separated_winning_set_n_split": sum(model_metadata(m)[1] == "split" for m in separated_members),
            "separated_winning_set_n_non_split": sum(model_metadata(m)[1] != "split" for m in separated_members),
            "separated_winning_set_composition": separated_comp,
        })
        for set_name, members in [("statistical_best_set", best_set), ("separated_winning_set", separated_members)]:
            for model in members:
                family, mode = model_metadata(model)
                set_member_rows.append({
                    **scenario, "metric": metric, "set_name": set_name, "model_label": model,
                    "model_family": family, "learning_mode": mode,
                })
    return {
        "scenario_results": pd.DataFrame(scenario_rows),
        "scenario_model_means": pd.DataFrame(model_mean_rows),
        "best_vs_all": pd.DataFrame(bva_rows),
        "all_pairwise": pd.DataFrame(pair_rows),
        "set_members": pd.DataFrame(set_member_rows),
    }


def summarize_wins(scenario_results: pd.DataFrame, set_members: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    model_rows = []
    for model in MODEL_ORDER:
        family, mode = model_metadata(model)
        model_rows.append({
            "model_label": model, "model_family": family, "learning_mode": mode,
            "n_scenarios": int(len(scenario_results)),
            "descriptive_wins": int(scenario_results["descriptive_best_model"].eq(model).sum()),
            "strict_statistical_wins": int(scenario_results["strict_winner_model"].eq(model).sum()),
            "statistical_best_set_memberships": int(((set_members["set_name"] == "statistical_best_set") & set_members["model_label"].eq(model)).sum()) if not set_members.empty else 0,
            "separated_winning_set_memberships": int(((set_members["set_name"] == "separated_winning_set") & set_members["model_label"].eq(model)).sum()) if not set_members.empty else 0,
        })
    model_summary = pd.DataFrame(model_rows)
    family_rows = []
    for family in sorted(set(model_summary["model_family"])):
        if set_members.empty:
            best_set_count = separated_count = 0
        else:
            best_set_count = set_members[set_members["set_name"].eq("statistical_best_set") & set_members["model_family"].eq(family)]["scenario_id"].nunique()
            separated_count = set_members[set_members["set_name"].eq("separated_winning_set") & set_members["model_family"].eq(family)]["scenario_id"].nunique()
        family_rows.append({
            "model_family": family, "n_scenarios": int(len(scenario_results)),
            "descriptive_wins": int(scenario_results["descriptive_best_family"].eq(family).sum()),
            "strict_statistical_wins": int(scenario_results["strict_winner_family"].eq(family).sum()),
            "statistical_best_set_memberships": int(best_set_count),
            "separated_winning_set_memberships": int(separated_count),
        })
    family_summary = pd.DataFrame(family_rows)
    categories = ["split_only_singleton", "split_only_multiple", "mixed_with_split", "non_split_only", "no_separated_set"]
    composition_summary = pd.DataFrame([{
        "set_composition": category,
        "statistical_best_set_scenarios": int(scenario_results["statistical_best_set_composition"].eq(category).sum()),
        "separated_winning_set_scenarios": int(scenario_results["separated_winning_set_composition"].eq(category).sum()),
    } for category in categories])
    return model_summary, family_summary, composition_summary


def agent_split_local_analysis(run_metrics: pd.DataFrame, metric: str = PRIMARY_METRIC) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    d = run_metrics[
        run_metrics["metric"].eq(metric)
        & run_metrics["level"].eq("agent")
        & run_metrics["value"].notna()
        & np.isfinite(run_metrics["value"])
    ].copy()
    detail_rows = []
    scenario_meta_cols = ["scenario_label", "lead_time", "num_products", "seasonality_frequency", "seasonality_magnitude", "noise_level"]
    for (scenario_id, family, agent_id), g in d.groupby(["scenario_id", "model_family", "agent_id"], dropna=False):
        if family not in FAMILY_PAIRS:
            continue
        local_model, split_model = FAMILY_PAIRS[family]
        pivot = g.pivot_table(index="run_id", columns="model_label", values="value", aggfunc="mean")
        if local_model not in pivot.columns or split_model not in pivot.columns:
            continue
        paired = pivot[[split_model, local_model]].dropna()
        test = paired_one_sided(paired[split_model], paired[local_model], lower_is_better=True)
        meta = {c: g.iloc[0][c] for c in scenario_meta_cols if c in g.columns}
        detail_rows.append({
            "scenario_id": scenario_id, **meta, "metric": metric, "model_family": family,
            "agent_id": agent_id, "split_model": split_model, "local_model": local_model,
            "split_mean_paired_runs": float(paired[split_model].mean()) if len(paired) else np.nan,
            "local_mean_paired_runs": float(paired[local_model].mean()) if len(paired) else np.nan,
            **test,
        })
    detail = pd.DataFrame(detail_rows)
    if detail.empty:
        return detail, pd.DataFrame(), pd.DataFrame()
    detail["p_adj_holm_agents"] = np.nan
    for _, idx in detail.groupby(["scenario_id", "model_family"]).groups.items():
        detail.loc[idx, "p_adj_holm_agents"] = holm_adjust(detail.loc[idx, "p_one_sided"])
    detail["test_available"] = np.isfinite(detail["p_one_sided"])
    detail["descriptive_split_improved"] = detail["mean_diff"] < 0
    detail["significant_split_improved"] = detail["descriptive_split_improved"] & (detail["p_adj_holm_agents"] <= ALPHA)

    expected_by_scenario = run_metrics.groupby("scenario_id")["expected_agent_count"].max().to_dict()
    sf_rows = []
    for (scenario_id, family), g in detail.groupby(["scenario_id", "model_family"]):
        expected = int(expected_by_scenario.get(scenario_id, g["agent_id"].nunique()))
        expected_ids = set(range(expected))
        tested_ids = set(pd.to_numeric(g["agent_id"], errors="coerce").dropna().astype(int))
        sf_rows.append({
            "scenario_id": scenario_id, "scenario_label": g.iloc[0].get("scenario_label"),
            "lead_time": g.iloc[0].get("lead_time"), "metric": metric, "model_family": family,
            "expected_agent_ids": ", ".join(map(str, sorted(expected_ids))),
            "tested_agent_ids": ", ".join(map(str, sorted(tested_ids))),
            "all_agents_tested": tested_ids == expected_ids,
            "all_agents_descriptively_improved": bool(tested_ids == expected_ids and g["descriptive_split_improved"].all()),
            "all_agents_significantly_improved": bool(tested_ids == expected_ids and g["significant_split_improved"].all()),
            "min_paired_runs_across_agents": int(g["n_pairs"].min()),
        })
    scenario_family = pd.DataFrame(sf_rows)
    family_summary = pd.DataFrame([{
        "model_family": family, "n_scenarios": len(g),
        "all_agents_tested_scenarios": int(g["all_agents_tested"].sum()),
        "all_agents_descriptively_improved_scenarios": int(g["all_agents_descriptively_improved"].sum()),
        "all_agents_significantly_improved_scenarios": int(g["all_agents_significantly_improved"].sum()),
    } for family, g in scenario_family.groupby("model_family")])
    return detail, scenario_family, family_summary


def build_tables(experiments_df: pd.DataFrame, run_metrics: pd.DataFrame) -> dict[str, pd.DataFrame]:
    experiments, metrics = enrich_old_pipeline_outputs(experiments_df, run_metrics)
    metric_summary = summarize_run_metrics(metrics)
    system = scenario_system_analysis(metrics)
    model_summary, family_summary, set_composition = summarize_wins(system["scenario_results"], system["set_members"])
    agent_detail, agent_scenario_family, agent_family_summary = agent_split_local_analysis(metrics)
    data_quality = pd.DataFrame([
        {"check": "original_experiment_instances_after_merge", "value": len(experiments), "status": "info"},
        {"check": "market_configuration_count", "value": experiments["scenario_id"].nunique(), "status": "info"},
        {"check": "complete_model_configurations", "value": int(experiments["complete_expected_runs"].sum()), "status": "ok" if experiments["complete_expected_runs"].all() else "warning"},
        {"check": "incomplete_model_configurations", "value": int((~experiments["complete_expected_runs"]).sum()), "status": "warning" if (~experiments["complete_expected_runs"]).any() else "ok"},
        {"check": "run_metric_rows", "value": len(metrics), "status": "info"},
        {"check": "source_level", "value": "Data/agent_sc_level_1.csv", "status": "info"},
        {"check": "read_merge_logic", "value": "preserved from original notebook", "status": "ok"},
    ])
    readme = pd.DataFrame([
        ["Purpose", "Summaries by market configuration; run_id is used only as the paired seed/block within each market configuration."],
        ["Read and merge logic", "The original discover_experiment_instances, config-signature validation, duplicate-run handling, merge_experiment_instances, iter_run_records_from_experiment_row and evaluate_experiment functions are preserved."],
        ["Source file per run", "Data/agent_sc_level_1.csv"],
        ["System MAE", "Sum of agent-level MAE values per run (system_sum_agents)."],
        ["Descriptive result", "Mean of each model's run-level system MAE within one market configuration; the lowest model is the descriptive winner."],
        ["Pairing", "Within a market configuration, identical run IDs are paired because they use the same market-demand seed."],
        ["Strict winner", "Descriptive best model must be significantly better than all competitors after Holm correction."],
        ["Statistical Best Set", "Best model plus all competitors not significantly worse in Holm-corrected best-vs-all tests; untestable comparisons remain conservatively in the set."],
        ["Separated Winning Set", "Smallest leading group internally not distinguishable and significantly separated from the next model, based on Holm-corrected all-pairwise tests."],
        ["Split set categories", "split_only_singleton; split_only_multiple; mixed_with_split; non_split_only; no_separated_set."],
        ["Agent analysis", "Split versus local is paired by market configuration, family, agent ID and run ID; Holm correction is applied across agents."],
        ["Expected run IDs", ", ".join(map(str, EXPECTED_RUN_IDS))],
        ["Alpha", ALPHA],
    ], columns=["key", "value"])
    availability_cols = [
        "scenario_id", "scenario_label", "experiment_folder", "model_label", "model_family", "learning_mode",
        "lead_time", "seasonality_frequency", "seasonality_magnitude", "noise_level", "n_runs_available",
        "run_ids_available", "run_ids_missing", "complete_expected_runs", "merged_from_n_instances",
        "n_reporting_roots", "source_reporting_roots", "source_experiment_paths", "config_signature_hash",
    ]
    availability = experiments[[c for c in availability_cols if c in experiments.columns]].sort_values(["scenario_id", "model_label"]).reset_index(drop=True)
    return {
        "00_README": readme,
        "01_Data_Quality": data_quality,
        "02_Run_Availability": availability,
        "03_Metric_Summary": metric_summary,
        "04_Scenario_Model_MAE": system["scenario_model_means"],
        "05_Scenario_Results_MAE": system["scenario_results"],
        "06_Model_Win_Summary": model_summary,
        "07_Family_Win_Summary": family_summary,
        "08_Set_Composition": set_composition,
        "09_Set_Members": system["set_members"],
        "10_Best_vs_All": system["best_vs_all"],
        "11_All_Pairwise": system["all_pairwise"],
        "12_Agent_Split_Local": agent_detail,
        "13_Agent_Scenario_Family": agent_scenario_family,
        "14_Agent_Family_Summary": agent_family_summary,
    }


def _excel_scalar(value: Any) -> Any:
    if isinstance(value, (list, tuple, set, dict)):
        return json.dumps(_normalize_json(value), ensure_ascii=False, default=str)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if pd.isna(value):
        return None
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    return value


def export_tables_to_excel(
    tables: dict[str, pd.DataFrame],
    output_path: Path,
) -> Path:
    """
    Exportiert alle DataFrames in eine formatierte Excel-Datei.

    Dieser Export verwendet ausschließlich pandas + openpyxl und benötigt
    daher kein artifact_tool. Leere Tabellen erhalten eine sichtbare
    Statuszeile statt eines vollständig leeren Sheets.
    """
    try:
        from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
        from openpyxl.worksheet.table import Table, TableStyleInfo
        from openpyxl.utils import get_column_letter
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Für den Excel-Export wird 'openpyxl' benötigt. "
            "Installiere es in der verwendeten Jupyter-Umgebung beispielsweise "
            "mit `%pip install openpyxl` und starte den Kernel anschließend neu."
        ) from exc

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    used_sheet_names: set[str] = set()
    used_table_names: set[str] = set()

    def unique_sheet_name(raw_name: str) -> str:
        base = re.sub(r"[\[\]:*?/\\]", "_", str(raw_name)).strip() or "Sheet"
        base = base[:31]
        candidate = base
        counter = 1
        while candidate in used_sheet_names:
            suffix = f"_{counter}"
            candidate = f"{base[:31 - len(suffix)]}{suffix}"
            counter += 1
        used_sheet_names.add(candidate)
        return candidate

    def unique_table_name(sheet_name: str) -> str:
        base = re.sub(r"[^A-Za-z0-9_]", "_", sheet_name)
        if not base or not base[0].isalpha():
            base = f"T_{base}"
        base = base[:240]
        candidate = base
        counter = 1
        while candidate in used_table_names:
            suffix = f"_{counter}"
            candidate = f"{base[:240 - len(suffix)]}{suffix}"
            counter += 1
        used_table_names.add(candidate)
        return candidate

    header_fill = PatternFill("solid", fgColor="17365D")
    header_font = Font(color="FFFFFF", bold=True)
    header_alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True,
    )
    body_alignment = Alignment(vertical="top", wrap_text=True)
    thin_gray = Side(style="thin", color="D9E1F2")
    cell_border = Border(
        left=thin_gray,
        right=thin_gray,
        top=thin_gray,
        bottom=thin_gray,
    )

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for raw_sheet_name, raw_df in tables.items():
            sheet_name = unique_sheet_name(raw_sheet_name)

            frame = raw_df.copy()
            if frame.empty:
                frame = pd.DataFrame(
                    {"Status": ["No rows produced for this table."]}
                )

            # Nicht-native Excel-Werte wie Listen, Dicts und numpy-Skalare
            # werden mit derselben Hilfslogik wie zuvor normalisiert.
            frame = frame.map(_excel_scalar)
            frame.to_excel(writer, sheet_name=sheet_name, index=False)

            worksheet = writer.book[sheet_name]
            worksheet.freeze_panes = "A2"
            worksheet.auto_filter.ref = worksheet.dimensions
            worksheet.sheet_view.showGridLines = False

            max_row = worksheet.max_row
            max_column = worksheet.max_column

            for cell in worksheet[1]:
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = header_alignment
                cell.border = cell_border

            worksheet.row_dimensions[1].height = 30

            for row in worksheet.iter_rows(
                min_row=2,
                max_row=max_row,
                min_col=1,
                max_col=max_column,
            ):
                for cell in row:
                    cell.alignment = body_alignment
                    cell.border = cell_border

            for column_index, column_name in enumerate(frame.columns, start=1):
                values = [str(column_name)]
                values.extend(
                    "" if value is None else str(value)
                    for value in frame.iloc[:, column_index - 1].tolist()
                )

                measured_width = max(
                    (len(value) for value in values),
                    default=10,
                ) + 2

                column_key = str(column_name).lower()
                preferred_cap = (
                    50
                    if any(
                        token in column_key
                        for token in [
                            "models",
                            "members",
                            "ids",
                            "source",
                            "file",
                            "path",
                            "config",
                            "value",
                        ]
                    )
                    else 24
                )

                worksheet.column_dimensions[
                    get_column_letter(column_index)
                ].width = min(max(measured_width, 11), preferred_cap)

            # Eine strukturierte Excel-Tabelle verbessert Filterbarkeit und
            # Lesbarkeit. Sie wird nur angelegt, wenn mindestens eine
            # Datenzeile vorhanden ist.
            if max_row >= 2 and max_column >= 1:
                table = Table(
                    displayName=unique_table_name(sheet_name),
                    ref=f"A1:{get_column_letter(max_column)}{max_row}",
                )
                table.tableStyleInfo = TableStyleInfo(
                    name="TableStyleMedium2",
                    showFirstColumn=False,
                    showLastColumn=False,
                    showRowStripes=True,
                    showColumnStripes=False,
                )
                # Structured Excel tables are intentionally disabled; AutoFilter remains active.

    return output_path


In [ ]:
# =============================================================================
# ERWEITERUNG: vollständige Analysen für MSE und operative Metriken
# =============================================================================
# Die ursprüngliche Einlese-, Merge- und Run-Metrik-Logik bleibt unverändert.
# Diese Zelle überschreibt nur die Auswertungs- und Excel-Exportfunktionen.

ANALYSIS_METRICS = ["MAE", "MSE", "BWR_order", "BWR_inventory", "IVR", "OVR"]


def agent_split_local_analysis(run_metrics: pd.DataFrame, metric: str = PRIMARY_METRIC):
    d = run_metrics[
        run_metrics["metric"].eq(metric)
        & run_metrics["level"].eq("agent")
        & run_metrics["value"].notna()
        & np.isfinite(run_metrics["value"])
    ].copy()
    lower = LOWER_IS_BETTER.get(metric, True)
    detail_rows = []
    scenario_meta_cols = ["scenario_label", "lead_time", "num_products", "seasonality_frequency", "seasonality_magnitude", "noise_level"]
    for (scenario_id, family, agent_id), g in d.groupby(["scenario_id", "model_family", "agent_id"], dropna=False):
        if family not in FAMILY_PAIRS:
            continue
        local_model, split_model = FAMILY_PAIRS[family]
        pivot = g.pivot_table(index="run_id", columns="model_label", values="value", aggfunc="mean")
        if local_model not in pivot.columns or split_model not in pivot.columns:
            continue
        paired = pivot[[split_model, local_model]].dropna()
        test = paired_one_sided(paired[split_model], paired[local_model], lower_is_better=lower)
        split_mean = float(paired[split_model].mean()) if len(paired) else np.nan
        local_mean = float(paired[local_model].mean()) if len(paired) else np.nan
        rel = ((local_mean - split_mean) / abs(local_mean) * 100.0) if lower and np.isfinite(local_mean) and local_mean != 0 else np.nan
        if not lower and np.isfinite(local_mean) and local_mean != 0:
            rel = (split_mean - local_mean) / abs(local_mean) * 100.0
        meta = {c: g.iloc[0][c] for c in scenario_meta_cols if c in g.columns}
        detail_rows.append({
            "scenario_id": scenario_id, **meta, "metric": metric, "model_family": family,
            "agent_id": agent_id, "split_model": split_model, "local_model": local_model,
            "split_mean_paired_runs": split_mean, "local_mean_paired_runs": local_mean,
            "relative_improvement_pct": rel, **test,
        })
    detail = pd.DataFrame(detail_rows)
    if detail.empty:
        return detail, pd.DataFrame(), pd.DataFrame()
    detail["p_adj_holm_agents"] = np.nan
    for _, idx in detail.groupby(["scenario_id", "model_family"]).groups.items():
        detail.loc[idx, "p_adj_holm_agents"] = holm_adjust(detail.loc[idx, "p_one_sided"])
    detail["test_available"] = np.isfinite(detail["p_one_sided"])
    detail["descriptive_split_improved"] = detail["mean_diff"] < 0 if lower else detail["mean_diff"] > 0
    detail["significant_split_improved"] = detail["descriptive_split_improved"] & (detail["p_adj_holm_agents"] <= ALPHA)
    expected_by_scenario = run_metrics.groupby("scenario_id")["expected_agent_count"].max().to_dict()
    sf_rows = []
    for (scenario_id, family), g in detail.groupby(["scenario_id", "model_family"]):
        expected = int(expected_by_scenario.get(scenario_id, g["agent_id"].nunique()))
        expected_ids = set(range(expected))
        tested_ids = set(pd.to_numeric(g["agent_id"], errors="coerce").dropna().astype(int))
        sf_rows.append({
            "scenario_id": scenario_id, "scenario_label": g.iloc[0].get("scenario_label"),
            "lead_time": g.iloc[0].get("lead_time"), "noise_level": g.iloc[0].get("noise_level"),
            "metric": metric, "model_family": family,
            "expected_agent_ids": ", ".join(map(str, sorted(expected_ids))),
            "tested_agent_ids": ", ".join(map(str, sorted(tested_ids))),
            "all_agents_tested": tested_ids == expected_ids,
            "all_agents_descriptively_improved": bool(tested_ids == expected_ids and g["descriptive_split_improved"].all()),
            "all_agents_significantly_improved": bool(tested_ids == expected_ids and g["significant_split_improved"].all()),
            "min_paired_runs_across_agents": int(g["n_pairs"].min()),
        })
    scenario_family = pd.DataFrame(sf_rows)
    family_summary = pd.DataFrame([{
        "metric": metric, "model_family": family, "n_scenarios": len(g),
        "all_agents_tested_scenarios": int(g["all_agents_tested"].sum()),
        "all_agents_descriptively_improved_scenarios": int(g["all_agents_descriptively_improved"].sum()),
        "all_agents_significantly_improved_scenarios": int(g["all_agents_significantly_improved"].sum()),
    } for family, g in scenario_family.groupby("model_family")])
    return detail, scenario_family, family_summary


def system_split_local_analysis(run_metrics: pd.DataFrame, metric: str, level: str = SYSTEM_LEVEL):
    d = run_metrics[
        run_metrics["metric"].eq(metric)
        & run_metrics["level"].eq(level)
        & run_metrics["value"].notna()
        & np.isfinite(run_metrics["value"])
    ].copy()
    lower = LOWER_IS_BETTER.get(metric, True)
    rows = []
    meta_cols = ["scenario_label", "lead_time", "num_products", "seasonality_frequency", "seasonality_magnitude", "noise_level"]
    for (scenario_id, family), g in d.groupby(["scenario_id", "model_family"], dropna=False):
        if family not in FAMILY_PAIRS:
            continue
        local_model, split_model = FAMILY_PAIRS[family]
        pivot = g.pivot_table(index="run_id", columns="model_label", values="value", aggfunc="mean")
        if local_model not in pivot.columns or split_model not in pivot.columns:
            continue
        paired = pivot[[split_model, local_model]].dropna()
        test = paired_one_sided(paired[split_model], paired[local_model], lower_is_better=lower)
        split_mean = float(paired[split_model].mean()) if len(paired) else np.nan
        local_mean = float(paired[local_model].mean()) if len(paired) else np.nan
        if np.isfinite(local_mean) and local_mean != 0:
            rel = ((local_mean - split_mean) if lower else (split_mean - local_mean)) / abs(local_mean) * 100.0
        else:
            rel = np.nan
        dz = test["mean_diff"] / test["diff_sd"] if np.isfinite(test["diff_sd"]) and test["diff_sd"] != 0 else np.nan
        meta = {c: g.iloc[0][c] for c in meta_cols if c in g.columns}
        rows.append({
            "scenario_id": scenario_id, **meta, "metric": metric, "level": level,
            "model_family": family, "split_model": split_model, "local_model": local_model,
            "split_mean_paired_runs": split_mean, "local_mean_paired_runs": local_mean,
            "relative_improvement_pct": rel, "cohens_dz": dz, **test,
        })
    detail = pd.DataFrame(rows)
    if detail.empty:
        return detail, pd.DataFrame()
    detail["p_adj_holm_families"] = np.nan
    for _, idx in detail.groupby("scenario_id").groups.items():
        detail.loc[idx, "p_adj_holm_families"] = holm_adjust(detail.loc[idx, "p_one_sided"])
    detail["test_available"] = np.isfinite(detail["p_one_sided"])
    detail["descriptive_split_improved"] = detail["mean_diff"] < 0 if lower else detail["mean_diff"] > 0
    detail["significant_split_improved"] = detail["descriptive_split_improved"] & (detail["p_adj_holm_families"] <= ALPHA)
    summary_rows = []
    specs = [
        ("overall", ["metric", "model_family"]),
        ("lead_time", ["metric", "model_family", "lead_time"]),
        ("lead_time_noise", ["metric", "model_family", "lead_time", "noise_level"]),
    ]
    for summary_level, cols in specs:
        for keys, g in detail.groupby(cols, dropna=False):
            if not isinstance(keys, tuple): keys = (keys,)
            row = dict(zip(cols, keys)); wins = g[g["descriptive_split_improved"]]; sig = g[g["significant_split_improved"]]
            row.update({
                "summary_level": summary_level, "n_scenarios": int(len(g)),
                "descriptive_split_wins": int(g["descriptive_split_improved"].sum()),
                "significant_split_wins": int(g["significant_split_improved"].sum()),
                "mean_relative_improvement_pct_all": float(g["relative_improvement_pct"].mean()),
                "median_relative_improvement_pct_all": float(g["relative_improvement_pct"].median()),
                "mean_relative_improvement_pct_wins": float(wins["relative_improvement_pct"].mean()) if len(wins) else np.nan,
                "median_relative_improvement_pct_wins": float(wins["relative_improvement_pct"].median()) if len(wins) else np.nan,
                "mean_relative_improvement_pct_significant": float(sig["relative_improvement_pct"].mean()) if len(sig) else np.nan,
                "median_relative_improvement_pct_significant": float(sig["relative_improvement_pct"].median()) if len(sig) else np.nan,
                "mean_abs_cohens_dz": float(g["cohens_dz"].abs().mean()),
            })
            summary_rows.append(row)
    return detail, pd.DataFrame(summary_rows)


def _concat_frames(frames):
    frames = [f for f in frames if isinstance(f, pd.DataFrame) and not f.empty]
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()


def _add_metric(frame, metric):
    frame = frame.copy()
    if "metric" not in frame.columns:
        frame.insert(0, "metric", metric)
    return frame


def build_tables(experiments_df: pd.DataFrame, run_metrics: pd.DataFrame):
    experiments, metrics = enrich_old_pipeline_outputs(experiments_df, run_metrics)
    metric_summary = summarize_run_metrics(metrics)
    available = set(metrics["metric"].dropna().astype(str))
    selected = [m for m in ANALYSIS_METRICS if m in available]
    systems = {}; model_sums=[]; family_sums=[]; comp_sums=[]; agent_details=[]; agent_scen=[]; agent_fam=[]; split_details=[]; split_sums=[]
    for metric in selected:
        system = scenario_system_analysis(metrics, metric=metric, level=SYSTEM_LEVEL); systems[metric] = system
        ms, fs, cs = summarize_wins(system["scenario_results"], system["set_members"])
        model_sums.append(_add_metric(ms, metric)); family_sums.append(_add_metric(fs, metric)); comp_sums.append(_add_metric(cs, metric))
        ad, asc, afs = agent_split_local_analysis(metrics, metric=metric)
        agent_details.append(_add_metric(ad, metric)); agent_scen.append(_add_metric(asc, metric)); agent_fam.append(_add_metric(afs, metric))
        sd, ss = system_split_local_analysis(metrics, metric=metric); split_details.append(sd); split_sums.append(ss)
    mae = systems[PRIMARY_METRIC]; mae_ms, mae_fs, mae_cs = summarize_wins(mae["scenario_results"], mae["set_members"]); mae_ad, mae_asc, mae_afs = agent_split_local_analysis(metrics, PRIMARY_METRIC)
    data_quality = pd.DataFrame([
        {"check":"original_experiment_instances_after_merge","value":len(experiments),"status":"info"},
        {"check":"market_configuration_count","value":experiments["scenario_id"].nunique(),"status":"info"},
        {"check":"complete_model_configurations","value":int(experiments["complete_expected_runs"].sum()),"status":"ok" if experiments["complete_expected_runs"].all() else "warning"},
        {"check":"incomplete_model_configurations","value":int((~experiments["complete_expected_runs"]).sum()),"status":"warning" if (~experiments["complete_expected_runs"]).any() else "ok"},
        {"check":"run_metric_rows","value":len(metrics),"status":"info"},
        {"check":"full_analysis_metrics","value":", ".join(selected),"status":"ok"},
    ])
    readme = pd.DataFrame([
        ["Purpose","Full paired scenario, Holm, set, agent and split-vs-local analyses for all configured metrics."],
        ["Metrics",", ".join(selected)], ["System level",SYSTEM_LEVEL],
        ["Direction","Lower is better for all selected metrics."],
        ["Relative improvement","Positive values mean that split improves on local."],
        ["Excel compatibility","Structured openpyxl Table objects are omitted; AutoFilter and formatting remain."],
        ["Expected run IDs",", ".join(map(str,EXPECTED_RUN_IDS))], ["Alpha",ALPHA],
    ], columns=["key","value"])
    availability_cols=["scenario_id","scenario_label","experiment_folder","model_label","model_family","learning_mode","lead_time","seasonality_frequency","seasonality_magnitude","noise_level","n_runs_available","run_ids_available","run_ids_missing","complete_expected_runs","merged_from_n_instances","n_reporting_roots","source_reporting_roots","source_experiment_paths","config_signature_hash"]
    availability=experiments[[c for c in availability_cols if c in experiments.columns]].sort_values(["scenario_id","model_label"]).reset_index(drop=True)
    return {
        "00_README":readme,"01_Data_Quality":data_quality,"02_Run_Availability":availability,"03_Metric_Summary":metric_summary,
        "04_Scenario_Model_MAE":mae["scenario_model_means"],"05_Scenario_Results_MAE":mae["scenario_results"],"06_Model_Win_Summary":mae_ms,"07_Family_Win_Summary":mae_fs,"08_Set_Composition":mae_cs,"09_Set_Members":mae["set_members"],"10_Best_vs_All":mae["best_vs_all"],"11_All_Pairwise":mae["all_pairwise"],"12_Agent_Split_Local":mae_ad,"13_Agent_Scenario_Family":mae_asc,"14_Agent_Family_Summary":mae_afs,
        "15_All_Scenario_Models":_concat_frames([s["scenario_model_means"] for s in systems.values()]),
        "16_All_Scenario_Results":_concat_frames([s["scenario_results"] for s in systems.values()]),
        "17_All_Model_Wins":_concat_frames(model_sums),"18_All_Family_Wins":_concat_frames(family_sums),"19_All_Set_Composition":_concat_frames(comp_sums),
        "20_All_Set_Members":_concat_frames([s["set_members"] for s in systems.values()]),"21_All_Best_vs_All":_concat_frames([s["best_vs_all"] for s in systems.values()]),"22_All_Pairwise":_concat_frames([s["all_pairwise"] for s in systems.values()]),
        "23_All_Agent_Split_Local":_concat_frames(agent_details),"24_All_Agent_Scen_Fam":_concat_frames(agent_scen),"25_All_Agent_Fam_Sum":_concat_frames(agent_fam),
        "26_All_System_Split_Local":_concat_frames(split_details),"27_All_Split_Local_Summary":_concat_frames(split_sums),
    }


def export_tables_to_excel(tables, output_path):
    """Safe openpyxl export without structured Table objects."""
    from openpyxl import load_workbook
    from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.utils import get_column_letter
    import os, tempfile, xml.etree.ElementTree as ET
    output_path=Path(output_path); output_path.parent.mkdir(parents=True, exist_ok=True)
    used=set()
    def sheet_name(raw):
        base=re.sub(r"[\[\]:*?/\\]","_",str(raw)).strip() or "Sheet"; base=base[:31]; name=base; i=1
        while name in used:
            suffix=f"_{i}"; name=base[:31-len(suffix)]+suffix; i+=1
        used.add(name); return name
    def safe(v):
        v=_excel_scalar(v)
        if isinstance(v,str): v=ILLEGAL_CHARACTERS_RE.sub("",v); v=v if len(v)<=32767 else v[:32764]+"..."
        return v
    hf=PatternFill("solid",fgColor="17365D"); hfont=Font(color="FFFFFF",bold=True); ha=Alignment(horizontal="center",vertical="center",wrap_text=True); ba=Alignment(vertical="top",wrap_text=True); side=Side(style="thin",color="D9E1F2"); border=Border(left=side,right=side,top=side,bottom=side)
    tmp=tempfile.NamedTemporaryFile(prefix=output_path.stem+"_",suffix=".xlsx",dir=output_path.parent,delete=False); tmp_path=Path(tmp.name); tmp.close()
    try:
        with pd.ExcelWriter(tmp_path,engine="openpyxl") as writer:
            for raw,df in tables.items():
                name=sheet_name(raw); frame=df.copy() if not df.empty else pd.DataFrame({"Status":["No rows produced."]}); frame=frame.apply(lambda c:c.map(safe)); frame.to_excel(writer,sheet_name=name,index=False); ws=writer.book[name]; ws.freeze_panes="A2"; ws.sheet_view.showGridLines=False; ws.auto_filter.ref=ws.dimensions
                for cell in ws[1]: cell.fill=hf; cell.font=hfont; cell.alignment=ha; cell.border=border
                ws.row_dimensions[1].height=30
                for row in ws.iter_rows(min_row=2,max_row=ws.max_row,min_col=1,max_col=ws.max_column):
                    for cell in row: cell.alignment=ba; cell.border=border
                for ci,col in enumerate(frame.columns,1):
                    key=str(col).lower(); cap=50 if any(t in key for t in ["models","members","ids","source","path","note","value"]) else 24; vals=[str(col)]+["" if v is None else str(v) for v in frame.iloc[:,ci-1].tolist()]; ws.column_dimensions[get_column_letter(ci)].width=min(max(max(map(len,vals))+2,11),cap)
        with zipfile.ZipFile(tmp_path) as zf:
            if zf.testzip() is not None: raise RuntimeError("Generated XLSX ZIP is corrupt")
            for member in zf.namelist():
                if member.endswith((".xml",".rels")): ET.fromstring(zf.read(member))
        check=load_workbook(tmp_path,read_only=True,data_only=False); check.close(); os.replace(tmp_path,output_path)
    except Exception:
        if tmp_path.exists(): tmp_path.unlink()
        raise
    return output_path

# =============================================================================
# ANALYSE-KONFIGURATION ALS JSON
# =============================================================================
def _json_unique_values(df: pd.DataFrame, column: str):
    values = _unique_non_missing_values(df, column)
    return [_normalize_json(value) for value in values]


def build_analysis_export_config(
    tables: dict,
    experiments: pd.DataFrame,
    metrics: pd.DataFrame,
):
    scenario_table = tables.get("05_Scenario_Results_MAE", pd.DataFrame()).copy()
    if scenario_table.empty:
        scenario_table = tables.get("16_All_Scenario_Results", pd.DataFrame()).copy()

    scenario_columns = [
        "scenario_id",
        "scenario_label",
        "lead_time",
        "num_products",
        "seasonality_frequency",
        "seasonality_magnitude",
        "noise_level",
        "lambda_value",
        "tau_value",
    ]
    scenario_columns = [column for column in scenario_columns if column in scenario_table.columns]

    if scenario_columns:
        market_scenarios_df = scenario_table[scenario_columns].drop_duplicates()
        scenario_sort_columns = [
            column for column in ["scenario_id", "lead_time"]
            if column in scenario_columns
        ]
        if scenario_sort_columns:
            market_scenarios_df = market_scenarios_df.sort_values(
                scenario_sort_columns,
                kind="stable",
            )
        market_scenarios_df = (
            market_scenarios_df
            .reset_index(drop=True)
            .astype(object)
            .where(pd.notna(market_scenarios_df), None)
        )
        market_scenarios = [
            _normalize_json(record)
            for record in market_scenarios_df.to_dict(orient="records")
        ]
    else:
        market_scenarios = []

    scenario_count = (
        int(scenario_table["scenario_id"].nunique())
        if "scenario_id" in scenario_table.columns
        else len(market_scenarios)
    )

    reporting_roots = [str(path) for path in resolve_reporting_roots()]
    experiment_folders = [str(value) for value in _unique_non_missing_values(experiments, "experiment_path")]
    source_configs = [str(value) for value in _unique_non_missing_values(experiments, "config_path")]

    return {
        "created_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "output": {
            "results_root": str(RESULTS_ROOT),
            "results_folder_name_requested": RESULTS_FOLDER_NAME,
            "results_folder_name_used": RESULTS_FOLDER_NAME_USED,
            "results_directory": str(EVALUATION_OUTPUT_DIR),
            "excel_file": str(OUTPUT_XLSX),
            "analysis_config_file": str(OUTPUT_CONFIG_JSON),
        },
        "input": {
            "reporting_roots": reporting_roots,
            "number_of_reporting_roots": len(reporting_roots),
            "experiment_folders": experiment_folders,
            "number_of_experiment_folders": len(experiment_folders),
            "source_config_files": source_configs,
        },
        "data": {
            "type": "synthetic" if DATA_IS_SYNTHETIC else "real_world",
            "synthetic": bool(DATA_IS_SYNTHETIC),
            "configured_data_sources": _json_unique_values(experiments, "data_source"),
        },
        "dependency_parameters": {
            "lambda": _json_unique_values(experiments, "lambda_value"),
            "tau": _json_unique_values(experiments, "tau_value"),
        },
        "scenarios": {
            "count": scenario_count,
            "market_scenarios": market_scenarios,
        },
        "evaluation_period": {
            "folder_tag_from_previous_logic": EVALUATION_WINDOW_TAG,
            "start": _json_unique_values(metrics, "test_start_time_used"),
            "length": _json_unique_values(metrics, "test_interval_used"),
            "end": _json_unique_values(metrics, "test_window_end_time_used"),
            "test_start_override": _normalize_json(TEST_START_TIME_OVERRIDE),
            "test_interval_override": _normalize_json(TEST_INTERVAL_OVERRIDE),
        },
        "training_period": {
            "training_time": _json_unique_values(experiments, "training_time"),
            "convergence_time": _json_unique_values(experiments, "convergence_time"),
            "simulation_time": _json_unique_values(experiments, "simulation_time"),
            "pre_training_end_used_as_evaluation_start": _json_unique_values(metrics, "test_start_time_used"),
        },
        "analysis": {
            "metrics": list(METRICS),
            "agent_level": AGENT_LEVEL,
            "system_view_level": SYSTEM_VIEW_LEVEL,
            "number_of_metric_rows": int(len(metrics)),
            "model_labels": _json_unique_values(experiments, "model_label"),
            "training_types": _json_unique_values(experiments, "training_type"),
        },
    }


def write_analysis_export_config(config_payload: dict, output_path: Path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(
            _normalize_json(config_payload),
            file,
            ensure_ascii=False,
            indent=2,
            default=str,
        )
    return output_path



## 7. Vollständige MSE- und Bullwhip-Auswertung mit kompatiblem Excel-Export

Die neue Auswertung erzeugt zusätzliche Sheets 15 bis 27. Der Export verzichtet auf strukturierte Excel-Tabellen, die den Reparaturdialog ausgelöst haben können.


In [ ]:
tables = build_tables(experiments_df, run_metrics)
excel_path = export_tables_to_excel(tables, OUTPUT_XLSX)
analysis_config = build_analysis_export_config(tables, experiments_df, run_metrics)
analysis_config_path = write_analysis_export_config(analysis_config, OUTPUT_CONFIG_JSON)

print(f"Excel saved: {OUTPUT_XLSX}")
print(f"Analysis config saved: {analysis_config_path}")
print(f"Sheets: {len(tables)}")
print(f"Metrics: {ANALYSIS_METRICS}")
print(f"Market configurations: {tables['05_Scenario_Results_MAE']['scenario_id'].nunique()}")
print(f"Run-metric rows: {len(run_metrics):,}")


## 8. Datenqualität und Run-Verfügbarkeit aus der Original-Merge-Pipeline

In [ ]:
display(tables['01_Data_Quality'])
display(tables['02_Run_Availability'])

## 9. Deskriptive Metrik-Summaries und Modellwerte je Marktkonfiguration

In [ ]:
display(tables['03_Metric_Summary'])
display(tables['04_Scenario_Model_MAE'].sort_values(['scenario_label', 'rank_by_mean']))

## 10. Zentrale Ergebniszeile je Marktkonfiguration

In [ ]:
display(tables['05_Scenario_Results_MAE'])

## 11. Häufigkeiten nach Modell und Modellfamilie

In [ ]:
display(tables['06_Model_Win_Summary'])
display(tables['07_Family_Win_Summary'])

## 12. Zusammensetzung und Mitglieder der statistischen Sets

In [ ]:
display(tables['08_Set_Composition'])
display(tables['09_Set_Members'])

## 13. Gepaarte Best-vs.-All- und vollständige paarweise Tests

In [ ]:
display(tables['10_Best_vs_All'])
display(tables['11_All_Pairwise'])

## 14. Split-vs.-Local auf Agentenebene

In [ ]:
display(tables['12_Agent_Split_Local'])
display(tables['13_Agent_Scenario_Family'])
display(tables['14_Agent_Family_Summary'])

## 15. Neue metrikenübergreifende Ergebnisse


In [ ]:
display(tables['16_All_Scenario_Results'])
display(tables['26_All_System_Split_Local'])
display(tables['27_All_Split_Local_Summary'])


## 15. Interpretation der Zähler

- `descriptive_wins`: In wie vielen Marktkonfigurationen hatte das Modell beziehungsweise die Familie den niedrigsten mittleren System-MAE?
- `strict_statistical_wins`: In wie vielen Marktkonfigurationen war der deskriptiv Beste nach Holm-Korrektur allein signifikant besser als alle anderen?
- `statistical_best_set_memberships`: In wie vielen Marktkonfigurationen gehörte das Modell beziehungsweise mindestens ein Modell der Familie zum Statistical Best Set?
- `separated_winning_set_memberships`: Entsprechende Mitgliedschaften im Separated Winning Set.
- `split_only_multiple`: Mehrere Split-Ansätze sind gemeinsam im besten Set, aber kein Non-Split-Ansatz.
- `mixed_with_split`: Mindestens ein Split- und mindestens ein Non-Split-Ansatz sind im Set.

Alle statistischen Tests verwenden nur identische Run-IDs innerhalb derselben Markt­konfiguration.